Setup

In [ ]:

# =============================================================================
# FINANCERAG-ANALYST — NOTEBOOK 5 PART 0: SETUP AND EVALUATION CONTRACT
# =============================================================================

from pathlib import Path
import json
import platform
import sys

import numpy as np
import pandas as pd
from google.colab import drive

# -----------------------------------------------------------------------------
# 0.1 — Mount Google Drive and define project paths
# -----------------------------------------------------------------------------

drive.mount("/content/drive")

REPO_ROOT = Path("/content/drive/MyDrive/FinanceRAG-Analyst")
ARTIFACTS_DIR = REPO_ROOT / "artifacts"
EVALUATION_DIR = ARTIFACTS_DIR / "evaluation"
GENERATION_DIR = ARTIFACTS_DIR / "generation"
GENERATION_EVALUATION_DIR = GENERATION_DIR / "evaluation"
GENERATION_TRACES_DIR = GENERATION_DIR / "traces"

print("=" * 80)
print("FINANCERAG-ANALYST — NOTEBOOK 5 PART 0: SETUP AND EVALUATION CONTRACT")
print("=" * 80)

print(f"Project root: {REPO_ROOT}")
print(f"Python: {sys.version.split()[0]}")
print(f"Platform: {platform.platform()}")
print(f"pandas: {pd.__version__}")
print(f"NumPy: {np.__version__}")

# -----------------------------------------------------------------------------
# 0.2 — Validate directory structure
# -----------------------------------------------------------------------------

required_directories = {
    "project root": REPO_ROOT,
    "artifacts directory": ARTIFACTS_DIR,
    "retrieval evaluation directory": EVALUATION_DIR,
    "generation directory": GENERATION_DIR,
    "generation evaluation directory": GENERATION_EVALUATION_DIR,
    "generation traces directory": GENERATION_TRACES_DIR,
}

print("\nDirectory validation:")
for name, path in required_directories.items():
    status = "PASS" if path.is_dir() else "FAIL"
    print(f"  [{status}] {name}: {path}")

missing_directories = [
    name for name, path in required_directories.items()
    if not path.is_dir()
]

if missing_directories:
    raise FileNotFoundError(
        "Required directories are missing: " + ", ".join(missing_directories)
    )

# -----------------------------------------------------------------------------
# 0.3 — Declare existing Notebook 3 inputs
# -----------------------------------------------------------------------------

N3_INPUTS = {
    "retrieval_summary":
        EVALUATION_DIR / "heldout_retrieval_summary_v2.json",

    "retrieval_config":
        EVALUATION_DIR / "heldout_retrieval_config_v2.json",

    "retrieval_runs":
        EVALUATION_DIR / "heldout_retrieval_runs_v2.parquet",

    "per_query_metrics":
        EVALUATION_DIR / "heldout_retrieval_per_query_metrics_v2.parquet",

    "heldout_queries":
        EVALUATION_DIR / "heldout_queries_v2.parquet",

    "human_judgments":
        EVALUATION_DIR / "heldout_human_judgments_final_v2.parquet",

    "human_judgments_manifest":
        EVALUATION_DIR / "heldout_human_judgments_final_v2_manifest.json",

    "qrels":
        EVALUATION_DIR / "heldout_qrels_final_v2.parquet",
}

# -----------------------------------------------------------------------------
# 0.4 — Declare actual Notebook 4 artifacts
# -----------------------------------------------------------------------------

N4_INPUTS = {
    "completion_manifest":
        GENERATION_EVALUATION_DIR / "notebook_04_completion_manifest_v1.json",

    "final_report":
        GENERATION_EVALUATION_DIR / "notebook_04_final_report_v1.json",

    "final_evaluation_manifest":
        GENERATION_EVALUATION_DIR
        / "notebook_04_groq_final_generation_evaluation_manifest_v1.json",

    "manual_scores":
        GENERATION_EVALUATION_DIR
        / "notebook_04_groq_manual_answer_scores_v1.parquet",

    "manual_scores_json":
        GENERATION_EVALUATION_DIR
        / "notebook_04_groq_manual_answer_scores_v1.json",

    "generation_summary":
        GENERATION_EVALUATION_DIR
        / "notebook_04_groq_final_generation_summary_v1.json",

    "corrected_citation_audit":
        GENERATION_EVALUATION_DIR
        / "notebook_04_groq_corrected_citation_audit_v2.parquet",

    "corrected_citation_manifest":
        GENERATION_EVALUATION_DIR
        / "notebook_04_groq_corrected_citation_manifest_v2.json",

    "artifact_inventory":
        GENERATION_EVALUATION_DIR
        / "notebook_04_final_artifact_inventory_v1.csv",

    "generation_traces":
        GENERATION_TRACES_DIR
        / "notebook_04_control_generations_groq_gpt_oss_120b_v2.json",
}

# -----------------------------------------------------------------------------
# 0.5 — Check declared artifact availability
# -----------------------------------------------------------------------------

def report_artifacts(title, artifacts):
    print(f"\n{title}:")
    missing = []

    for name, path in artifacts.items():
        exists = path.is_file()
        status = "FOUND" if exists else "NOT FOUND"
        print(f"  [{status}] {name}: {path}")

        if not exists:
            missing.append((name, path))

    return missing


missing_n3 = report_artifacts("Declared Notebook 3 inputs", N3_INPUTS)
missing_n4 = report_artifacts("Declared Notebook 4 inputs", N4_INPUTS)

# -----------------------------------------------------------------------------
# 0.6 — Read-only inspection of key Notebook 4 JSON manifests
# -----------------------------------------------------------------------------

JSON_TO_INSPECT = {
    name: N4_INPUTS[name]
    for name in [
        "completion_manifest",
        "final_report",
        "final_evaluation_manifest",
        "generation_summary",
        "corrected_citation_manifest",
    ]
}

print("\nNotebook 4 manifest inspection:")

for name, path in JSON_TO_INSPECT.items():
    if not path.is_file():
        print(f"\n[SKIPPED] {name}: file not found")
        continue

    print(f"\n--- {name}: {path.name} ---")

    try:
        with path.open("r", encoding="utf-8") as f:
            payload = json.load(f)

        if isinstance(payload, dict):
            print("Top-level keys:", list(payload.keys()))

            for key in [
                "artifact_name",
                "artifact_version",
                "status",
                "query_count",
                "created_at_utc",
            ]:
                if key in payload:
                    print(f"{key}: {payload[key]}")

            # Print headline metrics when available.
            for metric_key in ["headline_results", "headline_metrics"]:
                if metric_key in payload:
                    print(
                        json.dumps(
                            payload[metric_key],
                            indent=2,
                            ensure_ascii=False,
                        )[:5000]
                    )
        else:
            print("JSON type:", type(payload).__name__)

    except Exception as exc:
        print(f"[ERROR] Could not parse JSON: {type(exc).__name__}: {exc}")

# -----------------------------------------------------------------------------
# 0.7 — Freeze the Notebook 5 evaluation contract
# -----------------------------------------------------------------------------

EVALUATION_CONTRACT = {
    "project": "FinanceRAG-Analyst",
    "notebook": "05__end_to_end_rag_evaluation_and_deployment_readiness",
    "mode": "read_only_initialization",
    "preserve_notebook_3_artifacts": True,
    "preserve_notebook_4_artifacts": True,
    "make_llm_calls_during_setup": False,
    "overwrite_existing_artifacts_during_setup": False,
    "reuse_frozen_heldout_benchmark": True,
    "separate_new_robustness_tests": True,
    "validate_artifact_paths_before_loading": True,
}

print("\nEvaluation contract:")
print(json.dumps(EVALUATION_CONTRACT, indent=2))

print("\n" + "=" * 80)
print("PART 0 COMPLETE")
print("=" * 80)
print("No LLM calls were made.")
print("No existing evaluation artifacts were modified.")
print(f"Missing Notebook 3 inputs: {len(missing_n3)}")
print(f"Missing Notebook 4 inputs: {len(missing_n4)}")

if missing_n3 or missing_n4:
    print(
        "\nWARNING: Some declared files were not found. "
        "Resolve actual paths before proceeding to Part 1."
    )
else:
    print("\nAll declared input artifacts were found.")


Mounted at /content/drive
FINANCERAG-ANALYST — NOTEBOOK 5 PART 0: SETUP AND EVALUATION CONTRACT
Project root: /content/drive/MyDrive/FinanceRAG-Analyst
Python: 3.13.16
Platform: Linux-6.6.122+-x86_64-with-glibc2.39
pandas: 2.2.3
NumPy: 2.1.3

Directory validation:
  [PASS] project root: /content/drive/MyDrive/FinanceRAG-Analyst
  [PASS] artifacts directory: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts
  [PASS] retrieval evaluation directory: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation
  [PASS] generation directory: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation
  [PASS] generation evaluation directory: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/evaluation
  [PASS] generation traces directory: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/traces

Declared Notebook 3 inputs:
  [FOUND] retrieval_summary: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_summary_v2.json
  [FOUND] r

In [ ]:
# ============================================================
# FINANCERAG-ANALYST — NOTEBOOK 5 PART 1
# ARTIFACT DISCOVERY AND INTEGRITY VALIDATION
#
# Mode: Read-only
# LLM calls: None
# Existing artifact modifications: None
# ============================================================

from pathlib import Path
import json
import pandas as pd
import numpy as np

print("=" * 80)
print("NOTEBOOK 5 — PART 1: ARTIFACT DISCOVERY AND INTEGRITY")
print("=" * 80)


# ------------------------------------------------------------
# 1. Validate Part 0 setup
# ------------------------------------------------------------

required_globals = [
    "REPO_ROOT",
    "EVALUATION_DIR",
    "GENERATION_EVALUATION_DIR",
    "GENERATION_TRACES_DIR",
    "N3_INPUTS",
    "N4_INPUTS",
]

missing_globals = [
    name for name in required_globals
    if name not in globals()
]

if missing_globals:
    raise RuntimeError(
        "Part 0 setup is missing required variables: "
        + ", ".join(missing_globals)
        + ". Run the corrected Part 0 cell first."
    )

print("\n[PASS] Required Part 0 variables are available.")


# ------------------------------------------------------------
# 2. Initialize diagnostic containers
# ------------------------------------------------------------

integrity_results = []
loaded_json = {}
loaded_tables = {}

def record_result(name, status, detail):
    integrity_results.append({
        "artifact": name,
        "status": status,
        "detail": str(detail),
    })
    print(f"[{status}] {name}: {detail}")


# ------------------------------------------------------------
# 3. Read-only artifact loading helpers
# ------------------------------------------------------------

def load_json_artifact(name, path):
    """
    Load a JSON artifact without assuming its top-level
    structure is a dictionary.
    """
    path = Path(path)

    if not path.is_file():
        record_result(name, "FAIL", f"File not found: {path}")
        return None

    try:
        with path.open("r", encoding="utf-8") as f:
            obj = json.load(f)

        loaded_json[name] = obj

        if isinstance(obj, dict):
            detail = (
                f"Valid JSON dictionary; "
                f"{len(obj)} top-level keys"
            )
        elif isinstance(obj, list):
            detail = (
                f"Valid JSON list; "
                f"{len(obj)} top-level items"
            )
        else:
            detail = (
                f"Valid JSON; "
                f"top-level type={type(obj).__name__}"
            )

        record_result(name, "PASS", detail)
        return obj

    except Exception as exc:
        record_result(
            name,
            "FAIL",
            f"{type(exc).__name__}: {exc}"
        )
        return None


def load_table_artifact(name, path):
    """
    Load Parquet or CSV artifacts and report their schemas.
    """
    path = Path(path)

    if not path.is_file():
        record_result(name, "FAIL", f"File not found: {path}")
        return None

    try:
        suffix = path.suffix.lower()

        if suffix == ".parquet":
            df = pd.read_parquet(path)
        elif suffix == ".csv":
            df = pd.read_csv(path)
        else:
            record_result(
                name,
                "WARN",
                f"Unsupported table extension: {suffix}"
            )
            return None

        loaded_tables[name] = df

        record_result(
            name,
            "PASS",
            f"Loaded {len(df):,} rows × "
            f"{len(df.columns):,} columns"
        )

        print(f"    Columns: {df.columns.tolist()}")

        return df

    except Exception as exc:
        record_result(
            name,
            "FAIL",
            f"{type(exc).__name__}: {exc}"
        )
        return None


# ------------------------------------------------------------
# 4. Load Notebook 3 frozen retrieval artifacts
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("NOTEBOOK 3 — FROZEN RETRIEVAL ARTIFACTS")
print("-" * 80)

n3_json_keys = [
    "retrieval_summary",
    "retrieval_config",
    "human_judgments_manifest",
]

n3_table_keys = [
    "retrieval_runs",
    "per_query_metrics",
    "heldout_queries",
    "human_judgments",
    "qrels",
]

for key in n3_json_keys:
    load_json_artifact(
        f"N3::{key}",
        N3_INPUTS[key]
    )

for key in n3_table_keys:
    load_table_artifact(
        f"N3::{key}",
        N3_INPUTS[key]
    )


# ------------------------------------------------------------
# 5. Load Notebook 4 frozen generation artifacts
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("NOTEBOOK 4 — FROZEN GENERATION ARTIFACTS")
print("-" * 80)

n4_json_keys = [
    "completion_manifest",
    "final_report",
    "final_evaluation_manifest",
    "manual_scores_json",
    "generation_summary",
    "corrected_citation_manifest",
]

n4_table_keys = [
    "manual_scores",
    "corrected_citation_audit",
    "artifact_inventory",
]

for key in n4_json_keys:
    load_json_artifact(
        f"N4::{key}",
        N4_INPUTS[key]
    )

for key in n4_table_keys:
    load_table_artifact(
        f"N4::{key}",
        N4_INPUTS[key]
    )


# ------------------------------------------------------------
# 6. Load Notebook 4 generation traces
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("NOTEBOOK 4 — GENERATION TRACES")
print("-" * 80)

trace_path = Path(N4_INPUTS["generation_traces"])

generation_traces = None

if trace_path.is_file():
    try:
        with trace_path.open("r", encoding="utf-8") as f:
            generation_traces = json.load(f)

        loaded_json["N4::generation_traces"] = generation_traces

        if isinstance(generation_traces, (list, dict)):
            trace_count = len(generation_traces)
        else:
            trace_count = "not applicable"

        record_result(
            "N4::generation_traces",
            "PASS",
            f"Valid JSON; "
            f"type={type(generation_traces).__name__}; "
            f"top-level length={trace_count}"
        )

    except Exception as exc:
        record_result(
            "N4::generation_traces",
            "FAIL",
            f"{type(exc).__name__}: {exc}"
        )
else:
    record_result(
        "N4::generation_traces",
        "FAIL",
        f"File not found: {trace_path}"
    )


# ------------------------------------------------------------
# 7. Safe duplicate-row detection
# ------------------------------------------------------------

def safe_duplicate_count(df):
    """
    Prefer pandas duplicate detection.

    Some cells may contain lists, dictionaries, or NumPy arrays,
    which are unhashable. If pandas cannot compare the rows,
    fall back to a representation-based diagnostic.

    The fallback is not a canonical equality test. Later checks
    will validate duplicates using explicit business keys.
    """
    try:
        return int(df.duplicated().sum()), "pandas_exact"

    except (TypeError, ValueError):
        try:
            row_signatures = df.apply(
                lambda row: tuple(
                    repr(value)
                    for value in row.tolist()
                ),
                axis=1
            )

            return (
                int(row_signatures.duplicated().sum()),
                "repr_fallback"
            )

        except Exception as exc:
            return (
                None,
                f"unavailable: {type(exc).__name__}: {exc}"
            )


# ------------------------------------------------------------
# 8. Basic table integrity checks
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("BASIC TABLE INTEGRITY")
print("-" * 80)

for name, df in loaded_tables.items():

    missing_cells = int(
        df.isna().sum().sum()
    )

    duplicate_rows, duplicate_method = safe_duplicate_count(df)

    duplicate_display = (
        duplicate_rows
        if duplicate_rows is not None
        else "not computed"
    )

    record_result(
        f"{name}::basic_checks",
        "INFO",
        f"rows={len(df):,}; "
        f"columns={len(df.columns):,}; "
        f"missing_cells={missing_cells:,}; "
        f"duplicate_rows={duplicate_display}; "
        f"duplicate_method={duplicate_method}"
    )


# ------------------------------------------------------------
# 9. Inspect query identifier columns
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("QUERY IDENTIFIER DISCOVERY")
print("-" * 80)

query_id_candidates = {
    "query_id",
    "qid",
    "question_id",
    "heldout_query_id",
}

for name, df in loaded_tables.items():

    matching_columns = [
        col for col in df.columns
        if col.lower() in query_id_candidates
    ]

    if not matching_columns:
        record_result(
            f"{name}::query_id_discovery",
            "INFO",
            "No recognized query-ID column found"
        )
        continue

    for col in matching_columns:

        non_null = df[col].dropna()

        unique_count = int(
            non_null.nunique()
        )

        sample_values = (
            non_null.astype(str)
            .head(5)
            .tolist()
        )

        record_result(
            f"{name}::{col}",
            "INFO",
            f"non_null={len(non_null):,}; "
            f"unique={unique_count:,}; "
            f"sample={sample_values}"
        )


# ------------------------------------------------------------
# 10. Inspect required identifiers for key tables
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("REQUIRED KEY-COLUMN CHECKS")
print("-" * 80)

required_columns = {
    "N3::heldout_queries": [
        "query_id",
        "query",
    ],
    "N3::retrieval_runs": [
        "query_id",
        "retrieval_method",
        "rank",
        "chunk_id",
    ],
    "N3::per_query_metrics": [
        "query_id",
        "method",
    ],
    "N3::human_judgments": [
        "query_id",
        "chunk_id",
        "human_relevance",
    ],
    "N3::qrels": [
        "query_id",
        "chunk_id",
        "relevance_grade",
    ],
    "N4::manual_scores": [
        "query_id",
        "answer",
        "total_score_0_to_8",
        "acceptable_at_least_7_of_8",
        "material_failure",
    ],
    "N4::corrected_citation_audit": [
        "query_id",
        "normalized_citations",
        "citation_present",
        "invalid_citation_labels",
    ],
}

for artifact_name, required in required_columns.items():

    df = loaded_tables.get(artifact_name)

    if df is None:
        record_result(
            f"{artifact_name}::required_columns",
            "FAIL",
            "Table did not load"
        )
        continue

    missing_columns = [
        col for col in required
        if col not in df.columns
    ]

    if missing_columns:
        record_result(
            f"{artifact_name}::required_columns",
            "FAIL",
            f"Missing columns: {missing_columns}"
        )
    else:
        record_result(
            f"{artifact_name}::required_columns",
            "PASS",
            f"All {len(required)} required columns present"
        )


# ------------------------------------------------------------
# 11. Build artifact inventory for display
# ------------------------------------------------------------

print("\n" + "-" * 80)
print("LOADED ARTIFACT INVENTORY")
print("-" * 80)

inventory_rows = []

for name, obj in loaded_json.items():

    if isinstance(obj, (list, dict)):
        item_count = len(obj)
    else:
        item_count = None

    inventory_rows.append({
        "artifact": name,
        "type": "JSON",
        "rows_or_items": item_count,
        "columns": None,
    })

for name, df in loaded_tables.items():

    inventory_rows.append({
        "artifact": name,
        "type": "TABLE",
        "rows_or_items": len(df),
        "columns": len(df.columns),
    })

artifact_inventory_df = pd.DataFrame(
    inventory_rows,
    columns=[
        "artifact",
        "type",
        "rows_or_items",
        "columns",
    ]
)

if not artifact_inventory_df.empty:
    display(
        artifact_inventory_df
        .sort_values("artifact")
        .reset_index(drop=True)
    )
else:
    print("No artifacts loaded successfully.")


# ------------------------------------------------------------
# 12. Final integrity summary
# ------------------------------------------------------------

print("\n" + "=" * 80)
print("PART 1 SUMMARY")
print("=" * 80)

integrity_report_df = pd.DataFrame(
    integrity_results,
    columns=[
        "artifact",
        "status",
        "detail",
    ]
)

if not integrity_report_df.empty:

    status_counts = (
        integrity_report_df["status"]
        .value_counts()
        .to_dict()
    )

    print("Status counts:", status_counts)

    failures = integrity_report_df[
        integrity_report_df["status"] == "FAIL"
    ]

    if not failures.empty:
        print("\nFailures requiring attention:")
        display(
            failures.reset_index(drop=True)
        )
    else:
        print("No artifact-loading or required-column failures detected.")

    warnings = integrity_report_df[
        integrity_report_df["status"] == "WARN"
    ]

    if not warnings.empty:
        print("\nWarnings:")
        display(
            warnings.reset_index(drop=True)
        )

else:
    print("No integrity results were generated.")

print("\nRead-only execution complete.")
print("Source artifacts modified: NO")
print("LLM calls made: NO")
print(
    "Cross-artifact query alignment and metric reproduction "
    "are not yet certified by this cell."
)

print("=" * 80)
print("PART 1 CELL COMPLETE")
print("=" * 80)

NOTEBOOK 5 — PART 1: ARTIFACT DISCOVERY AND INTEGRITY

[PASS] Required Part 0 variables are available.

--------------------------------------------------------------------------------
NOTEBOOK 3 — FROZEN RETRIEVAL ARTIFACTS
--------------------------------------------------------------------------------
[PASS] N3::retrieval_summary: Valid JSON dictionary; 10 top-level keys
[PASS] N3::retrieval_config: Valid JSON dictionary; 18 top-level keys
[PASS] N3::human_judgments_manifest: Valid JSON dictionary; 8 top-level keys
[PASS] N3::retrieval_runs: Loaded 360 rows × 35 columns
    Columns: ['rank', 'retrieval_method', 'retrieval_score', 'bm25_rank', 'dense_rank', 'corpus_row', 'chunk_id', 'document_id', 'chunk_index', 'start_span_index', 'end_span_index', 'start_char', 'end_char', 'token_count', 'below_minimum_target', 'text', 'ticker', 'company', 'cik', 'form_type', 'filing_date', 'report_date', 'accession_number', 'primary_document', 'source_url', 'raw_content_sha256', 'processed_content

,artifact,type,rows_or_items,columns
0,N3::heldout_queries,TABLE,12,11.0
1,N3::human_judgments,TABLE,214,27.0
2,N3::human_judgments_manifest,JSON,8,NaN
3,N3::per_query_metrics,TABLE,36,26.0
4,N3::qrels,TABLE,214,8.0
5,N3::retrieval_config,JSON,18,NaN
6,N3::retrieval_runs,TABLE,360,35.0
7,N3::retrieval_summary,JSON,10,NaN
8,N4::artifact_inventory,TABLE,8,4.0
9,N4::completion_manifest,JSON,10,NaN



PART 1 SUMMARY
Status counts: {'PASS': 25, 'INFO': 16}
No artifact-loading or required-column failures detected.

Read-only execution complete.
Source artifacts modified: NO
LLM calls made: NO
Cross-artifact query alignment and metric reproduction are not yet certified by this cell.
PART 1 CELL COMPLETE


In [ ]:

# ============================================================
# NOTEBOOK 5 — PART 2A
# Cross-artifact query alignment and schema inspection
#
# Prerequisites: Notebook 5 Parts 0 and 1 have run.
# Read-only: no source artifacts modified, no LLM calls,
# and no files written.
# ============================================================

import json
import pandas as pd
from pathlib import Path

# Confirm Part 0 variables exist.
required_variables = [
    "REPO_ROOT",
    "N3_INPUTS",
    "N4_INPUTS",
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    raise RuntimeError(
        "Run Notebook 5 Part 0 first. Missing variables: "
        + ", ".join(missing_variables)
    )

# ------------------------------------------------------------
# 1. Load the existing artifacts read-only
# ------------------------------------------------------------

def load_parquet_artifact(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label}: {path}")
    return pd.read_parquet(path)

def load_json_artifact(path, label):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"{label}: {path}")
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

queries = load_parquet_artifact(
    N3_INPUTS["heldout_queries"], "Held-out queries"
)
per_query_metrics = load_parquet_artifact(
    N3_INPUTS["per_query_metrics"], "Per-query retrieval metrics"
)
retrieval_runs = load_parquet_artifact(
    N3_INPUTS["retrieval_runs"], "Retrieval runs"
)
human_judgments = load_parquet_artifact(
    N3_INPUTS["human_judgments"], "Human judgments"
)
qrels = load_parquet_artifact(
    N3_INPUTS["qrels"], "Qrels"
)
generation_scores = load_parquet_artifact(
    N4_INPUTS["manual_scores"], "Generation scores"
)
citation_audit = load_parquet_artifact(
    N4_INPUTS["corrected_citation_audit"], "Citation audit"
)
generation_traces = load_json_artifact(
    N4_INPUTS["generation_traces"], "Generation traces"
)

# ------------------------------------------------------------
# 2. Print schemas and candidate query-ID columns
# ------------------------------------------------------------

frames = {
    "queries": queries,
    "per_query_metrics": per_query_metrics,
    "retrieval_runs": retrieval_runs,
    "human_judgments": human_judgments,
    "qrels": qrels,
    "generation_scores": generation_scores,
    "citation_audit": citation_audit,
}

print("=" * 76)
print("PART 2A — SCHEMA INSPECTION")
print("=" * 76)

for name, df in frames.items():
    print(f"\n{name}: shape={df.shape}")
    print("Columns:", list(df.columns))

    candidate_id_cols = [
        c for c in df.columns
        if any(token in c.lower() for token in
               ["query_id", "question_id", "case_id", "qid"])
    ]
    print("Candidate ID columns:", candidate_id_cols)

    for c in candidate_id_cols:
        values = df[c].dropna().astype(str).unique().tolist()
        print(f"  {c}: {len(values)} unique non-null values; "
              f"sample={values[:15]}")

# ------------------------------------------------------------
# 3. Inspect method labels without assuming their names
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("RETRIEVAL METHOD LABELS")
print("=" * 76)

for name in ["per_query_metrics", "retrieval_runs"]:
    df = frames[name]
    candidate_method_cols = [
        c for c in df.columns
        if any(token in c.lower() for token in
               ["method", "retriever", "strategy", "approach"])
    ]
    print(f"\n{name}:")
    if not candidate_method_cols:
        print("  No obvious method column detected.")
    for c in candidate_method_cols:
        print(f"  {c}: {df[c].value_counts(dropna=False).to_dict()}")

# ------------------------------------------------------------
# 4. Inspect trace identifiers
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("GENERATION TRACE STRUCTURE")
print("=" * 76)

print("Trace container type:", type(generation_traces).__name__)
print("Trace count:", len(generation_traces)
      if hasattr(generation_traces, "__len__") else "unknown")

if isinstance(generation_traces, list) and generation_traces:
    print("First trace keys:", list(generation_traces[0].keys())
          if isinstance(generation_traces[0], dict)
          else type(generation_traces[0]).__name__)

    for i, trace in enumerate(generation_traces[:3]):
        if isinstance(trace, dict):
            print(f"\nTrace {i} identifier-like fields:")
            for key, value in trace.items():
                if any(token in key.lower() for token in
                       ["query", "question", "case", "id"]):
                    print(f"  {key}: {str(value)[:180]}")

# ------------------------------------------------------------
# 5. Identify query-ID columns conservatively
# ------------------------------------------------------------

def find_query_id_column(df, label):
    preferred = [
        "query_id",
        "question_id",
        "qid",
        "heldout_query_id",
    ]
    lower_to_actual = {c.lower(): c for c in df.columns}

    for candidate in preferred:
        if candidate in lower_to_actual:
            return lower_to_actual[candidate]

    matches = [
        c for c in df.columns
        if "query_id" in c.lower()
        or "question_id" in c.lower()
    ]

    if len(matches) == 1:
        return matches[0]

    print(
        f"WARNING: Could not safely identify one query-ID column "
        f"for {label}. Candidates={matches}"
    )
    return None

print("\n" + "=" * 76)
print("QUERY-ID ALIGNMENT")
print("=" * 76)

id_columns = {}
query_id_sets = {}

for name, df in frames.items():
    id_col = find_query_id_column(df, name)
    id_columns[name] = id_col

    if id_col is None:
        continue

    ids = set(df[id_col].dropna().astype(str).str.strip())
    query_id_sets[name] = ids

    print(
        f"{name}: {len(ids)} unique IDs; "
        f"null IDs={int(df[id_col].isna().sum())}"
    )

# The held-out query table is the initial reference set.
reference_name = "queries"

if id_columns[reference_name] is None:
    raise RuntimeError(
        "Could not safely identify the held-out query ID column. "
        "Inspect the schema output before proceeding."
    )

reference_ids = query_id_sets[reference_name]

print(f"\nReference query count: {len(reference_ids)}")
print("Reference IDs:", sorted(reference_ids))

for name, ids in query_id_sets.items():
    if name == reference_name:
        continue

    missing = sorted(reference_ids - ids)
    extra = sorted(ids - reference_ids)

    print(f"\n{name}:")
    print(f"  Missing reference IDs: {missing}")
    print(f"  Extra IDs: {extra}")
    print(
        "  Query-ID set aligned:",
        not missing and not extra
    )

# ------------------------------------------------------------
# 6. Summarize the alignment findings
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("PART 2A STATUS")
print("=" * 76)

alignment_results = {}

for name, ids in query_id_sets.items():
    missing = sorted(reference_ids - ids)
    extra = sorted(ids - reference_ids)
    alignment_results[name] = {
        "unique_query_ids": len(ids),
        "missing_reference_ids": missing,
        "extra_ids": extra,
        "set_aligned": not missing and not extra,
    }

all_aligned = all(
    result["set_aligned"]
    for name, result in alignment_results.items()
    if name != reference_name
)

print("All tabular query-ID sets aligned:", all_aligned)
print("No files written.")
print("No source artifacts modified.")
print("No LLM calls made.")

# Keep inspection objects available for Part 2B.
PART2_FRAMES = frames
PART2_ID_COLUMNS = id_columns
PART2_QUERY_ID_SETS = query_id_sets
PART2_ALIGNMENT_RESULTS = alignment_results
PART2_REFERENCE_IDS = reference_ids
PART2_GENERATION_TRACES = generation_traces


PART 2A — SCHEMA INSPECTION

queries: shape=(12, 11)
Columns: ['query_id', 'ticker', 'report_year', 'question_type', 'difficulty', 'query', 'sec_filing_year', 'filing_date', 'report_date', 'document_id', 'accession_number']
Candidate ID columns: ['query_id']
  query_id: 12 unique non-null values; sample=['H01', 'H02', 'H03', 'H04', 'H05', 'H06', 'H07', 'H08', 'H09', 'H10', 'H11', 'H12']

per_query_metrics: shape=(36, 26)
Columns: ['query_id', 'method', 'pooled_candidates', 'pooled_useful', 'pooled_direct', 'retrieved_irrelevant_at_10', 'retrieved_partial_at_10', 'retrieved_direct_at_10', 'useful_precision_at_5', 'useful_precision_at_10', 'useful_pool_recall_at_5', 'useful_pool_recall_at_10', 'useful_hit_at_5', 'useful_hit_at_10', 'useful_mrr_at_10', 'direct_precision_at_5', 'direct_precision_at_10', 'direct_pool_recall_at_5', 'direct_pool_recall_at_10', 'direct_hit_at_5', 'direct_hit_at_10', 'direct_mrr_at_10', 'graded_ndcg_at_5', 'graded_ndcg_at_10', 'first_useful_rank', 'first_direct

In [ ]:

# ============================================================
# NOTEBOOK 5 — PART 2B
# Unified end-to-end evaluation dataset
#
# Read-only source artifacts. In-memory outputs only.
# No LLM calls. No files written.
# ============================================================

import pandas as pd
import numpy as np

assert "PART2_FRAMES" in globals(), "Run Part 2A first."
assert "PART2_ALIGNMENT_RESULTS" in globals(), "Run Part 2A first."

frames = PART2_FRAMES
queries = frames["queries"].copy()
metrics = frames["per_query_metrics"].copy()
runs = frames["retrieval_runs"].copy()
judgments = frames["human_judgments"].copy()
qrels = frames["qrels"].copy()
scores = frames["generation_scores"].copy()
audit = frames["citation_audit"].copy()
traces = PART2_GENERATION_TRACES

checks = []

def record_check(name, passed, detail):
    checks.append({
        "check": name,
        "status": "PASS" if bool(passed) else "REVIEW",
        "detail": str(detail),
    })

def normalize_id(series):
    return series.astype("string").str.strip()

# ------------------------------------------------------------
# 1. Confirm uniqueness and expected row counts
# ------------------------------------------------------------

record_check(
    "Held-out query IDs unique",
    queries["query_id"].notna().all()
    and not queries["query_id"].duplicated().any(),
    f"{len(queries)} rows; "
    f"{queries['query_id'].nunique()} unique query IDs"
)

expected_methods = {"BM25", "Dense", "Hybrid RRF"}

actual_methods = set(metrics["method"].dropna().astype(str).unique())

record_check(
    "Retrieval methods match expected benchmark methods",
    actual_methods == expected_methods,
    f"Observed={sorted(actual_methods)}"
)

metric_counts = metrics.groupby("query_id")["method"].nunique()

record_check(
    "Three retrieval methods per query",
    len(metric_counts) == len(queries)
    and metric_counts.eq(3).all()
    and not metrics.duplicated(["query_id", "method"]).any(),
    f"Method-count distribution={metric_counts.value_counts().to_dict()}"
)

run_counts = runs.groupby(
    ["query_id", "evaluation_method"]
).size()

expected_run_methods = {"bm25", "dense", "hybrid_rrf"}
actual_run_methods = set(
    runs["evaluation_method"].dropna().astype(str).unique()
)

record_check(
    "Retrieval-run methods recognized",
    actual_run_methods == expected_run_methods,
    f"Observed={sorted(actual_run_methods)}"
)

record_check(
    "Ten retrieved rows per query and evaluation method",
    len(run_counts) == len(queries) * 3 and run_counts.eq(10).all(),
    f"Rows per query/method distribution="
    f"{run_counts.value_counts().to_dict()}"
)

for name, df in [
    ("generation scores", scores),
    ("citation audit", audit),
]:
    passed = (
        df["query_id"].notna().all()
        and not df["query_id"].duplicated().any()
    )
    record_check(
        f"{name}: one row per query",
        passed,
        f"{len(df)} rows; {df['query_id'].nunique()} unique IDs"
    )

# ------------------------------------------------------------
# 2. Compare human judgments and qrels by evidence key
# ------------------------------------------------------------

candidate_keys = ["query_id", "corpus_row", "chunk_id"]
join_keys = [c for c in candidate_keys
             if c in judgments.columns and c in qrels.columns]

if len(join_keys) == len(candidate_keys):
    judgment_keys = judgments[join_keys].copy()
    qrel_keys = qrels[join_keys].copy()

    judgment_keys = judgment_keys.astype("string").fillna("<NULL>")
    qrel_keys = qrel_keys.astype("string").fillna("<NULL>")

    judgment_duplicates = int(judgment_keys.duplicated().sum())
    qrel_duplicates = int(qrel_keys.duplicated().sum())

    judgment_set = set(map(tuple, judgment_keys.to_numpy()))
    qrel_set = set(map(tuple, qrel_keys.to_numpy()))

    missing_from_qrels = judgment_set - qrel_set
    missing_from_judgments = qrel_set - judgment_set

    record_check(
        "Judgment/qrels evidence keys align",
        judgment_duplicates == 0
        and qrel_duplicates == 0
        and not missing_from_qrels
        and not missing_from_judgments,
        f"Keys={join_keys}; "
        f"judgment duplicates={judgment_duplicates}; "
        f"qrels duplicates={qrel_duplicates}; "
        f"judgments absent from qrels={len(missing_from_qrels)}; "
        f"qrels absent from judgments={len(missing_from_judgments)}"
    )

    if "human_relevance" in judgments.columns and \
       "relevance_grade" in qrels.columns and \
       not missing_from_qrels and not missing_from_judgments:
        compare = judgments[join_keys + ["human_relevance"]].merge(
            qrels[join_keys + ["relevance_grade"]],
            on=join_keys,
            how="inner",
            validate="one_to_one"
        )
        label_match = (
            compare["human_relevance"].astype("string").fillna("<NULL>")
            == compare["relevance_grade"].astype("string").fillna("<NULL>")
        )
        record_check(
            "Human relevance labels match qrels grades",
            label_match.all(),
            f"Compared={len(compare)}; mismatches={int((~label_match).sum())}"
        )
else:
    record_check(
        "Judgment/qrels evidence keys align",
        False,
        f"Could not identify all composite keys. Found={join_keys}"
    )

# ------------------------------------------------------------
# 3. Check query text consistency
# ------------------------------------------------------------

query_text_col = "query"
generation_text_col = "query_text"

if query_text_col in queries.columns and \
   generation_text_col in scores.columns:
    qtext = queries[["query_id", query_text_col]].merge(
        scores[["query_id", generation_text_col]],
        on="query_id",
        how="left",
        validate="one_to_one"
    )
    text_match = (
        qtext[query_text_col].astype("string").str.strip()
        == qtext[generation_text_col].astype("string").str.strip()
    )
    record_check(
        "Benchmark and generation query text match",
        text_match.all(),
        f"Compared={len(qtext)}; mismatches={int((~text_match).sum())}"
    )

# ------------------------------------------------------------
# 4. Validate generation trace IDs and metadata
# ------------------------------------------------------------

trace_df = pd.DataFrame(traces) if isinstance(traces, list) else pd.DataFrame()

if "query_id" in trace_df.columns:
    trace_ids = normalize_id(trace_df["query_id"])
    reference_ids = set(normalize_id(queries["query_id"]).dropna())
    observed_trace_ids = set(trace_ids.dropna())

    record_check(
        "Generation trace IDs align with benchmark",
        not trace_ids.isna().any()
        and not trace_ids.duplicated().any()
        and observed_trace_ids == reference_ids,
        f"Trace rows={len(trace_df)}; unique IDs={len(observed_trace_ids)}; "
        f"missing={sorted(reference_ids - observed_trace_ids)}; "
        f"extra={sorted(observed_trace_ids - reference_ids)}"
    )
else:
    record_check(
        "Generation trace IDs align with benchmark",
        False,
        "Trace objects do not expose a query_id column."
    )

# ------------------------------------------------------------
# 5. Compare context hashes across generation artifacts
# ------------------------------------------------------------

hash_frames = {
    "generation_scores": scores,
    "citation_audit": audit,
}

if "context_sha256" in trace_df.columns:
    hash_frames["generation_traces"] = trace_df

hash_table = None

for name, df in hash_frames.items():
    if "query_id" not in df.columns or "context_sha256" not in df.columns:
        record_check(
            f"{name}: context hashes available",
            False,
            "Required query_id/context_sha256 columns are missing."
        )
        continue

    tmp = df[["query_id", "context_sha256"]].copy()
    tmp["query_id"] = normalize_id(tmp["query_id"])
    tmp = tmp.rename(columns={"context_sha256": f"{name}_hash"})

    if hash_table is None:
        hash_table = tmp
    else:
        hash_table = hash_table.merge(
            tmp,
            on="query_id",
            how="outer",
            validate="one_to_one"
        )

if hash_table is not None:
    hash_columns = [c for c in hash_table.columns if c.endswith("_hash")]

    if len(hash_columns) >= 2:
        hash_complete = hash_table[hash_columns].notna().all(axis=1)
        hash_same = hash_table[hash_columns].nunique(axis=1, dropna=True).eq(1)

        record_check(
            "Generation context hashes consistent",
            hash_complete.all() and hash_same.all(),
            f"Queries compared={len(hash_table)}; "
            f"incomplete hash rows={int((~hash_complete).sum())}; "
            f"hash disagreement rows={int((~hash_same).sum())}"
        )

# ------------------------------------------------------------
# 6. Create query-level retrieval metric columns
# ------------------------------------------------------------

metric_value_cols = [
    c for c in metrics.columns
    if c not in ["query_id", "method"]
]

method_name_map = {
    "BM25": "bm25",
    "Dense": "dense",
    "Hybrid RRF": "hybrid_rrf",
}

metrics_for_wide = metrics.copy()
metrics_for_wide["method_key"] = metrics_for_wide["method"].map(
    method_name_map
)

if metrics_for_wide["method_key"].isna().any():
    raise ValueError(
        "Unmapped retrieval method labels found; refusing to pivot."
    )

wide_metrics = metrics_for_wide.pivot(
    index="query_id",
    columns="method_key",
    values=metric_value_cols
)

wide_metrics.columns = [
    f"{method}_{metric}"
    for metric, method in wide_metrics.columns
]
wide_metrics = wide_metrics.reset_index()
wide_metrics.columns.name = None

# ------------------------------------------------------------
# 7. Create a compact trace summary
# ------------------------------------------------------------

trace_summary_cols = [
    "query_id",
    "status",
    "retrieval_method",
    "provider",
    "model",
    "latency_seconds",
    "retrieved_evidence_count",
    "selected_evidence_count",
    "context_sha256",
]

available_trace_cols = [
    c for c in trace_summary_cols if c in trace_df.columns
]

trace_summary = trace_df[available_trace_cols].copy()

rename_trace = {
    "status": "trace_status",
    "retrieval_method": "generation_retrieval_method",
    "provider": "generation_provider",
    "model": "generation_model",
    "latency_seconds": "generation_latency_seconds",
    "retrieved_evidence_count": "trace_retrieved_evidence_count",
    "selected_evidence_count": "trace_selected_evidence_count",
    "context_sha256": "trace_context_sha256",
}

trace_summary = trace_summary.rename(columns=rename_trace)

# ------------------------------------------------------------
# 8. Merge the unified query-level dataset
# ------------------------------------------------------------

query_base_cols = [
    c for c in [
        "query_id", "ticker", "report_year", "question_type",
        "difficulty", "query", "document_id", "accession_number",
        "filing_date", "report_date"
    ] if c in queries.columns
]

score_cols = [
    c for c in scores.columns
    if c != "query_text"
]

audit_cols = [
    c for c in audit.columns
    if c not in scores.columns or c == "query_id"
]

unified = queries[query_base_cols].merge(
    wide_metrics,
    on="query_id",
    how="left",
    validate="one_to_one"
)

unified = unified.merge(
    scores[score_cols],
    on="query_id",
    how="left",
    validate="one_to_one",
    suffixes=("", "_score")
)

unified = unified.merge(
    audit[audit_cols],
    on="query_id",
    how="left",
    validate="one_to_one",
    suffixes=("", "_citation")
)

unified = unified.merge(
    trace_summary,
    on="query_id",
    how="left",
    validate="one_to_one"
)

record_check(
    "Unified dataset has one row per held-out query",
    len(unified) == len(queries)
    and unified["query_id"].nunique() == len(queries),
    f"Unified shape={unified.shape}"
)

# ------------------------------------------------------------
# 9. Print results and keep outputs in memory
# ------------------------------------------------------------

check_df = pd.DataFrame(checks)

print("=" * 76)
print("PART 2B — VALIDATION CHECKS")
print("=" * 76)
print(check_df.to_string(index=False))

print("\n" + "=" * 76)
print("UNIFIED DATASET")
print("=" * 76)
print("Shape:", unified.shape)
print("Unique queries:", unified["query_id"].nunique())
print("Columns:")
print(unified.columns.tolist())

display_cols = [
    c for c in [
        "query_id", "ticker", "report_year", "question_type",
        "bm25_useful_mrr_at_10",
        "dense_useful_mrr_at_10",
        "hybrid_rrf_useful_mrr_at_10",
        "total_score_rate", "acceptable_at_least_7_of_8",
        "material_failure", "citation_format_pass"
    ] if c in unified.columns
]

print("\nQuery-level overview:")
display(unified[display_cols].sort_values("query_id"))

print("\nCheck status counts:")
print(check_df["status"].value_counts().to_dict())

print("\nNo files written. No source artifacts modified. No LLM calls.")

# Keep outputs available for Part 3.
PART2B_UNIFIED = unified
PART2B_CHECKS = check_df
PART2B_TRACE_SUMMARY = trace_summary
PART2B_HASH_TABLE = hash_table


PART 2B — VALIDATION CHECKS
                                             check status                                                                                                                                               detail
                         Held-out query IDs unique   PASS                                                                                                                         12 rows; 12 unique query IDs
Retrieval methods match expected benchmark methods   PASS                                                                                                             Observed=['BM25', 'Dense', 'Hybrid RRF']
                 Three retrieval methods per query   PASS                                                                                                                    Method-count distribution={3: 12}
                  Retrieval-run methods recognized   PASS                                                                                       

,query_id,ticker,report_year,question_type,bm25_useful_mrr_at_10,dense_useful_mrr_at_10,hybrid_rrf_useful_mrr_at_10,total_score_rate,acceptable_at_least_7_of_8,material_failure,citation_format_pass
0,H01,AAPL,2025,reported_driver,1.000000,1.000000,1.0,0.875,True,False,True
1,H02,AAPL,2024,geographic_or_segment_comparison,1.000000,1.000000,1.0,1.000,True,False,True
2,H03,MSFT,2025,segment_driver,0.125000,1.000000,0.5,1.000,True,False,True
3,H04,MSFT,2024,capital_expenditure,1.000000,0.500000,0.5,0.875,True,False,True
4,H05,AMZN,2024,segment_comparison,1.000000,1.000000,1.0,1.000,True,False,True
5,H06,AMZN,2023,cash_flow_driver,0.250000,0.500000,0.5,1.000,True,False,True
6,H07,GOOGL,2024,revenue_driver,1.000000,0.200000,0.5,1.000,True,False,True
7,H08,GOOGL,2023,risk_factor,0.250000,0.333333,1.0,1.000,True,False,True
8,H09,NVDA,2025,segment_driver,1.000000,1.000000,1.0,1.000,True,False,True
9,H10,NVDA,2024,supply_risk,0.500000,1.000000,1.0,1.000,True,False,True



Check status counts:
{'PASS': 13}

No files written. No source artifacts modified. No LLM calls.


In [ ]:

# ============================================================
# NOTEBOOK 5 — PART 3A
# Reproduce aggregate retrieval and generation metrics
#
# Uses frozen Notebook 3 and Notebook 4 outputs.
# No LLM calls. No files written. No source modifications.
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

assert "PART2B_UNIFIED" in globals(), "Run Part 2B first."

frames = PART2_FRAMES
metrics = frames["per_query_metrics"].copy()
scores = frames["generation_scores"].copy()
audit = frames["citation_audit"].copy()

# ------------------------------------------------------------
# 1. Retrieval metric reproduction
# ------------------------------------------------------------

retrieval_metric_columns = [
    "useful_mrr_at_10",
    "graded_ndcg_at_10",
    "useful_precision_at_10",
    "useful_pool_recall_at_10",
    "direct_mrr_at_10",
    "graded_ndcg_at_5",
]

available_retrieval_metrics = [
    c for c in retrieval_metric_columns if c in metrics.columns
]

retrieval_summary = (
    metrics.groupby("method", dropna=False)[available_retrieval_metrics]
    .agg(["mean", "median", "std"])
    .round(6)
)

print("=" * 76)
print("PART 3A — RETRIEVAL METRIC REPRODUCTION")
print("=" * 76)
print("Queries:", metrics["query_id"].nunique())
print("Rows:", len(metrics))
print("Methods:", sorted(metrics["method"].dropna().unique().tolist()))
display(retrieval_summary)

# Concise comparison against the previously reported headline metrics.
headline_metrics = [
    "useful_mrr_at_10",
    "graded_ndcg_at_10",
]

headline_rows = []
for method in ["BM25", "Dense", "Hybrid RRF"]:
    subset = metrics.loc[metrics["method"] == method]
    row = {"method": method, "query_count": subset["query_id"].nunique()}
    for col in headline_metrics:
        if col in subset.columns:
            row[f"mean_{col}"] = subset[col].mean()
    headline_rows.append(row)

retrieval_headline = pd.DataFrame(headline_rows)

print("\nHeadline retrieval results (mean over per-query metrics):")
display(retrieval_headline.round(6))

# ------------------------------------------------------------
# 2. Generation metric reproduction
# ------------------------------------------------------------

score_columns = [
    "answer_correctness_score_0_to_2",
    "evidence_support_score_0_to_2",
    "answer_completeness_score_0_to_2",
    "citation_correctness_score_0_to_2",
    "total_score_0_to_8",
]

for col in score_columns:
    if col in scores.columns:
        scores[col] = pd.to_numeric(scores[col], errors="coerce")

n = len(scores)

def pct(numerator, denominator):
    return (
        100.0 * numerator / denominator
        if denominator else np.nan
    )

generation_summary = {
    "query_count": n,
    "total_score_points": float(scores["total_score_0_to_8"].sum()),
    "total_score_possible": int(8 * n),
    "mean_score_out_of_8": float(scores["total_score_0_to_8"].mean()),
    "mean_score_rate_pct": float(
        100 * scores["total_score_0_to_8"].mean() / 8
    ),
    "answer_correctness_pct_of_max": pct(
        scores["answer_correctness_score_0_to_2"].sum(), 2 * n
    ),
    "evidence_support_pct_of_max": pct(
        scores["evidence_support_score_0_to_2"].sum(), 2 * n
    ),
    "completeness_pct_of_max": pct(
        scores["answer_completeness_score_0_to_2"].sum(), 2 * n
    ),
    "citation_correctness_pct_of_max": pct(
        scores["citation_correctness_score_0_to_2"].sum(), 2 * n
    ),
    "fully_passed_count": int(scores["fully_passed"].fillna(False).astype(bool).sum()),
    "acceptable_count": int(
        scores["acceptable_at_least_7_of_8"].fillna(False).astype(bool).sum()
    ),
    "material_failure_count": int(
        scores["material_failure"].fillna(False).astype(bool).sum()
    ),
}

generation_summary["fully_passed_pct"] = pct(
    generation_summary["fully_passed_count"], n
)
generation_summary["acceptable_pct"] = pct(
    generation_summary["acceptable_count"], n
)
generation_summary["material_failure_pct"] = pct(
    generation_summary["material_failure_count"], n
)

print("\n" + "=" * 76)
print("GENERATION QUALITY METRICS")
print("=" * 76)

for key, value in generation_summary.items():
    if isinstance(value, (float, np.floating)):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

# ------------------------------------------------------------
# 3. Citation-format reproduction
# ------------------------------------------------------------

citation_boolean_cols = [
    "citation_present",
    "citation_format_pass",
    "parser_result_changed",
]

citation_summary = {
    "query_count": len(audit),
    "queries_with_citations": int(
        audit["citation_present"].fillna(False).astype(bool).sum()
    ),
    "citation_format_pass_count": int(
        audit["citation_format_pass"].fillna(False).astype(bool).sum()
    ),
    "parser_result_changed_count": int(
        audit["parser_result_changed"].fillna(False).astype(bool).sum()
    ),
}

citation_summary["citation_presence_pct"] = pct(
    citation_summary["queries_with_citations"], len(audit)
)
citation_summary["citation_format_pass_pct"] = pct(
    citation_summary["citation_format_pass_count"], len(audit)
)

if "invalid_citation_labels" in audit.columns:
    citation_summary["invalid_label_occurrences"] = int(
        pd.to_numeric(
            audit["invalid_citation_labels"], errors="coerce"
        ).fillna(0).sum()
    )

print("\n" + "=" * 76)
print("CITATION AUDIT METRICS")
print("=" * 76)

for key, value in citation_summary.items():
    if isinstance(value, (float, np.floating)):
        print(f"{key}: {value:.4f}")
    else:
        print(f"{key}: {value}")

# ------------------------------------------------------------
# 4. Failure breakdown
# ------------------------------------------------------------

failure_cols = [
    c for c in [
        "query_id", "ticker", "failure_category", "review_notes",
        "total_score_0_to_8", "acceptable_at_least_7_of_8",
        "material_failure",
    ]
    if c in PART2B_UNIFIED.columns
]

print("\n" + "=" * 76)
print("GENERATION FAILURE / REVIEW BREAKDOWN")
print("=" * 76)

if "material_failure" in scores.columns:
    material_ids = set(
        scores.loc[
            scores["material_failure"].fillna(False).astype(bool),
            "query_id"
        ].astype(str)
    )
    print("Material failure query IDs:", sorted(material_ids))

if "failure_category" in scores.columns:
    print(
        "Failure-category counts (including blank/none):",
        scores["failure_category"].fillna("<NULL>").value_counts().to_dict()
    )

display(PART2B_UNIFIED[failure_cols].sort_values("query_id"))

# ------------------------------------------------------------
# 5. Inspect existing N3/N4 summary artifacts for comparison
# ------------------------------------------------------------

print("\n" + "=" * 76)
print("EXISTING SUMMARY ARTIFACTS — STRUCTURE INSPECTION")
print("=" * 76)

summary_paths = {}

if "summary" in N3_INPUTS:
    summary_paths["N3 retrieval summary"] = N3_INPUTS["summary"]

if "final_report" in N4_INPUTS:
    summary_paths["N4 final report"] = N4_INPUTS["final_report"]

for label, raw_path in summary_paths.items():
    path = Path(raw_path)
    print(f"\n{label}: {path}")

    if not path.exists():
        print("  File not found.")
        continue

    try:
        with path.open("r", encoding="utf-8") as f:
            obj = json.load(f)

        if isinstance(obj, dict):
            print("  Top-level keys:", list(obj.keys()))
            print("  JSON preview:")
            print(json.dumps(obj, indent=2, ensure_ascii=False)[:5000])
        else:
            print("  JSON type:", type(obj).__name__)
            print("  JSON preview:")
            print(json.dumps(obj, indent=2, ensure_ascii=False)[:2000])

    except Exception as exc:
        print(f"  Could not read summary: {type(exc).__name__}: {exc}")

# ------------------------------------------------------------
# 6. Keep derived results in memory for Part 3B
# ------------------------------------------------------------

PART3A_RETRIEVAL_SUMMARY = retrieval_summary
PART3A_RETRIEVAL_HEADLINE = retrieval_headline
PART3A_GENERATION_SUMMARY = generation_summary
PART3A_CITATION_SUMMARY = citation_summary

print("\nNo files written.")
print("No source artifacts modified.")
print("No LLM calls made.")


PART 3A — RETRIEVAL METRIC REPRODUCTION
Queries: 12
Rows: 36
Methods: ['BM25', 'Dense', 'Hybrid RRF']


useful_mrr_at_10                  graded_ndcg_at_10            \
                       mean median       std              mean    median   
method                                                                     
BM25               0.690972   1.00  0.391859          0.511835  0.464999   
Dense              0.681349   0.75  0.350413          0.582095  0.604205   
Hybrid RRF         0.750000   1.00  0.337100          0.598620  0.621158   

                     useful_precision_at_10                   \
                 std                   mean median       std   
method                                                         
BM25        0.275696               0.391667   0.40  0.242930   
Dense       0.239088               0.508333   0.55  0.253909   
Hybrid RRF  0.238121               0.441667   0.45  0.287492   

           useful_pool_recall_at_10                     direct_mrr_at_10  \
                               mean    median       std             mean   
method                                                                     
BM25                       0.568575  0.568182  0.256746         0.494444   
Dense                      0.745579  0.775000  0.248296         0.451025   
Hybrid RRF                 0.607082  0.645833  0.247338         0.673611   

                               graded_ndcg_at_5                      
              median       std             mean    median       std  
method                                                               
BM25        0.291667  0.386798         0.426631  0.378540  0.317747  
Dense       0.500000  0.302776         0.475971  0.507536  0.266742  
Hybrid RRF  0.750000  0.366629         0.558861  0.573972  0.239951


Headline retrieval results (mean over per-query metrics):


,method,query_count,mean_useful_mrr_at_10,mean_graded_ndcg_at_10
0,BM25,12,0.690972,0.511835
1,Dense,12,0.681349,0.582095
2,Hybrid RRF,12,0.750000,0.598620



GENERATION QUALITY METRICS
query_count: 12
total_score_points: 88.0000
total_score_possible: 96
mean_score_out_of_8: 7.3333
mean_score_rate_pct: 91.6667
answer_correctness_pct_of_max: 87.5000
evidence_support_pct_of_max: 95.8333
completeness_pct_of_max: 87.5000
citation_correctness_pct_of_max: 95.8333
fully_passed_count: 9
acceptable_count: 11
material_failure_count: 1
fully_passed_pct: 75.0000
acceptable_pct: 91.6667
material_failure_pct: 8.3333

CITATION AUDIT METRICS
query_count: 12
queries_with_citations: 12
citation_format_pass_count: 12
parser_result_changed_count: 4
citation_presence_pct: 100.0000
citation_format_pass_pct: 100.0000
invalid_label_occurrences: 0

GENERATION FAILURE / REVIEW BREAKDOWN
Material failure query IDs: ['H12']
Failure-category counts (including blank/none): {'none': 9, 'incomplete_answer': 1, 'minor_claim_conflation': 1, 'retrieval_led_answer_failure': 1}


,query_id,ticker,failure_category,review_notes,total_score_0_to_8,acceptable_at_least_7_of_8,material_failure
0,H01,AAPL,incomplete_answer,The answer accurately reports the retrieved Ma...,7,True,False
1,H02,AAPL,none,Correctly reports the change and 2024 net-sale...,8,True,False
2,H03,MSFT,none,Correctly identifies Azure as the principal In...,8,True,False
3,H04,MSFT,minor_claim_conflation,"The facilities, datacenters, computer systems,...",7,True,False
4,H05,AMZN,none,Correctly reports AWS net sales and operating ...,8,True,False
5,H06,AMZN,none,Correctly identifies improved net income after...,8,True,False
6,H07,GOOGL,none,"Correctly distinguishes the drivers of Search,...",8,True,False
7,H08,GOOGL,none,"Provides a broad, evidence-supported account o...",8,True,False
8,H09,NVDA,none,Correctly attributes Data Center growth to Hop...,8,True,False
9,H10,NVDA,none,Comprehensively and accurately summarizes foun...,8,True,False



EXISTING SUMMARY ARTIFACTS — STRUCTURE INSPECTION

N4 final report: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/evaluation/notebook_04_final_report_v1.json
  Top-level keys: ['notebook', 'status', 'evaluation_scope', 'headline_results', 'principal_successes', 'principal_failures', 'conclusion', 'known_limitations', 'next_engineering_priorities']
  JSON preview:
{
  "notebook": "04_generation_and_grounding_evaluation",
  "status": "COMPLETE",
  "evaluation_scope": {
    "held_out_query_count": 12,
    "generation_provider": "groq",
    "generation_model": "groq/openai/gpt-oss-120b",
    "generation_policy_version": "groq_gpt_oss_120b_v2",
    "manual_reviewers": 1,
    "manual_score_range_per_dimension": [
      0,
      2
    ],
    "manual_dimensions": [
      "answer correctness",
      "evidence support",
      "answer completeness",
      "citation correctness"
    ]
  },
  "headline_results": {
    "generation_completion_count": 12,
    "generation_completion_r

In [ ]:

# ============================================================
# NOTEBOOK 5 — PART 3B
# Resolve MRR metric definitions and locate N3 summary
# Read-only; no LLM calls; no files written.
# ============================================================

import json
from pathlib import Path
import pandas as pd

metrics = PART2_FRAMES["per_query_metrics"]

print("RETRIEVAL METRIC MEANS")
metric_cols = [
    c for c in [
        "useful_mrr_at_10",
        "direct_mrr_at_10",
        "graded_ndcg_at_10",
        "useful_precision_at_10",
        "useful_pool_recall_at_10",
    ]
    if c in metrics.columns
]

display(
    metrics.groupby("method")[metric_cols]
    .mean()
    .reindex(["BM25", "Dense", "Hybrid RRF"])
    .round(6)
)

print("\nN3 INPUT KEYS")
print(sorted(N3_INPUTS.keys()))

print("\nN4 INPUT KEYS")
print(sorted(N4_INPUTS.keys()))

print("\nCANDIDATE SUMMARY FILES")
root = Path(REPO_ROOT)

candidate_paths = [
    root / "artifacts/evaluation/heldout_retrieval_summary_v2.json",
    root / "artifacts/evaluation/heldout_retrieval_config_v2.json",
    root / "artifacts/generation/evaluation/notebook_04_final_report_v1.json",
]

for path in candidate_paths:
    print(f"\n{path}")
    if not path.exists():
        print("  NOT FOUND")
        continue

    try:
        with path.open("r", encoding="utf-8") as f:
            obj = json.load(f)

        print("  Top-level keys:", list(obj.keys())
              if isinstance(obj, dict) else type(obj).__name__)

        if "summary" in path.name:
            print(json.dumps(obj, indent=2, ensure_ascii=False)[:10000])
        elif "config" in path.name:
            print(json.dumps(obj, indent=2, ensure_ascii=False)[:4000])
        elif "final_report" in path.name and isinstance(obj, dict):
            print(json.dumps(
                obj.get("headline_results", {}),
                indent=2,
                ensure_ascii=False
            ))
    except Exception as exc:
        print(f"  ERROR: {type(exc).__name__}: {exc}")

print("\nNo files written. Source artifacts unchanged.")


RETRIEVAL METRIC MEANS


,useful_mrr_at_10,direct_mrr_at_10,graded_ndcg_at_10,useful_precision_at_10,useful_pool_recall_at_10
method,,,,,
BM25,0.690972,0.494444,0.511835,0.391667,0.568575
Dense,0.681349,0.451025,0.582095,0.508333,0.745579
Hybrid RRF,0.750000,0.673611,0.598620,0.441667,0.607082



N3 INPUT KEYS
['heldout_queries', 'human_judgments', 'human_judgments_manifest', 'per_query_metrics', 'qrels', 'retrieval_config', 'retrieval_runs', 'retrieval_summary']

N4 INPUT KEYS
['artifact_inventory', 'completion_manifest', 'corrected_citation_audit', 'corrected_citation_manifest', 'final_evaluation_manifest', 'final_report', 'generation_summary', 'generation_traces', 'manual_scores', 'manual_scores_json']

CANDIDATE SUMMARY FILES

/content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_summary_v2.json
  Top-level keys: ['evaluation_version', 'question_count', 'judgment_count', 'methods', 'primary_metric_definitions', 'recall_scope', 'bootstrap', 'aggregate_primary_metrics', 'paired_primary_comparisons', 'failure_summary']
{
  "evaluation_version": "heldout_v2",
  "question_count": 12,
  "judgment_count": 214,
  "methods": [
    "BM25",
    "Dense",
    "Hybrid RRF"
  ],
  "primary_metric_definitions": {
    "useful_pool_recall_at_10": "Useful pool Reca

In [ ]:

# ============================================================
# PART 3C — FORMAL METRIC RECONCILIATION
# Read-only; no LLM calls; no source artifact modifications.
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

# Reuse the validated artifacts and frames from Parts 0–3.
n3_summary_path = REPO_ROOT / "artifacts/evaluation/heldout_retrieval_summary_v2.json"
n4_report_path = (
    REPO_ROOT
    / "artifacts/generation/evaluation/notebook_04_final_report_v1.json"
)

assert n3_summary_path.exists(), f"Missing frozen N3 summary: {n3_summary_path}"
assert n4_report_path.exists(), f"Missing frozen N4 report: {n4_report_path}"

with open(n3_summary_path, "r") as f:
    n3_summary = json.load(f)

with open(n4_report_path, "r") as f:
    n4_report = json.load(f)

metrics = PART2_FRAMES["per_query_metrics"].copy()

scores = pd.read_parquet(N4_INPUTS["manual_scores"])

citation_audit = pd.read_parquet(
    N4_INPUTS["corrected_citation_audit"]
)

# ---- 1. Recompute retrieval metrics from per-query values ----

retrieval_metric_columns = [
    "direct_mrr_at_10",
    "useful_mrr_at_10",
    "graded_ndcg_at_10",
    "useful_precision_at_10",
    "useful_pool_recall_at_10",
]

recomputed_retrieval = (
    metrics.groupby("method")[retrieval_metric_columns]
    .mean()
    .reset_index()
)

# Frozen N3 aggregate primary metrics.
frozen_rows = n3_summary["aggregate_primary_metrics"]
frozen_lookup = {
    (row["method"], row["metric"]): row["macro_mean"]
    for row in frozen_rows
}

reconciliation_rows = []

for _, row in recomputed_retrieval.iterrows():
    method = row["method"]

    for metric_name in retrieval_metric_columns:
        recomputed_value = float(row[metric_name])
        frozen_value = frozen_lookup.get((method, metric_name))

        if frozen_value is None:
            reconciliation_rows.append({
                "source": "N3",
                "method": method,
                "metric": metric_name,
                "recomputed": recomputed_value,
                "frozen_report": np.nan,
                "absolute_difference": np.nan,
                "status": "NOT_IN_FROZEN_HEADLINE",
            })
        else:
            difference = abs(recomputed_value - float(frozen_value))
            reconciliation_rows.append({
                "source": "N3",
                "method": method,
                "metric": metric_name,
                "recomputed": recomputed_value,
                "frozen_report": float(frozen_value),
                "absolute_difference": difference,
                "status": "MATCH" if np.isclose(
                    recomputed_value, float(frozen_value),
                    atol=1e-8, rtol=1e-7
                ) else "MISMATCH",
            })

retrieval_reconciliation = pd.DataFrame(reconciliation_rows)

print("RETRIEVAL METRIC RECONCILIATION")
display(
    retrieval_reconciliation.sort_values(["metric", "method"])
    .round(8)
    .reset_index(drop=True)
)

# ---- 2. Recompute N4 generation metrics ----

score_columns = {
    "answer_correctness_score_0_to_2": "answer_correctness_score_rate",
    "evidence_support_score_0_to_2": "evidence_support_score_rate",
    "answer_completeness_score_0_to_2": "answer_completeness_score_rate",
    "citation_correctness_score_0_to_2": "citation_correctness_score_rate",
}

headline = n4_report["headline_results"]

# Inspect the report's exact nesting before relying on it.
print("\nN4 HEADLINE RESULTS STRUCTURE")
print("Type:", type(headline).__name__)
if isinstance(headline, dict):
    print("Keys:", list(headline.keys()))

# The saved report may store metrics directly or under a nested key.
if isinstance(headline, dict):
    frozen_n4 = headline
else:
    frozen_n4 = {}

n4_reconciliation_rows = []

n = len(scores)
assert n > 0, "No generation scores loaded."

for score_col, report_key in score_columns.items():
    recomputed_value = float(scores[score_col].mean()) / 2.0
    frozen_value = frozen_n4.get(report_key)

    n4_reconciliation_rows.append({
        "metric": report_key,
        "recomputed": recomputed_value,
        "frozen_report": (
            float(frozen_value)
            if isinstance(frozen_value, (int, float))
            else np.nan
        ),
        "status": (
            "MATCH"
            if isinstance(frozen_value, (int, float))
            and np.isclose(
                recomputed_value, float(frozen_value),
                atol=1e-8, rtol=1e-7
            )
            else "CHECK_REPORT_NESTING"
        ),
    })

recomputed_total = float(scores["total_score_0_to_8"].sum())
recomputed_max = float(n * 8)
recomputed_score_rate = recomputed_total / recomputed_max

for metric_name, recomputed_value, report_key in [
    ("overall_manual_rubric_score_rate", recomputed_score_rate,
     "overall_manual_rubric_score_rate"),
    ("acceptable_answer_rate",
     float(scores["acceptable_at_least_7_of_8"].astype(bool).mean()),
     "acceptable_answer_rate"),
    ("fully_passing_answer_rate",
     float(scores["fully_passed"].astype(bool).mean()),
     "fully_passing_answer_rate"),
    ("material_failure_rate",
     float(scores["material_failure"].astype(bool).mean()),
     "material_failure_rate"),
]:
    frozen_value = frozen_n4.get(report_key)
    n4_reconciliation_rows.append({
        "metric": metric_name,
        "recomputed": recomputed_value,
        "frozen_report": (
            float(frozen_value)
            if isinstance(frozen_value, (int, float))
            else np.nan
        ),
        "status": (
            "MATCH"
            if isinstance(frozen_value, (int, float))
            and np.isclose(
                recomputed_value, float(frozen_value),
                atol=1e-8, rtol=1e-7
            )
            else "CHECK_REPORT_NESTING"
        ),
    })

n4_reconciliation = pd.DataFrame(n4_reconciliation_rows)

print("\nGENERATION METRIC RECONCILIATION")
display(n4_reconciliation.round(8))

# ---- 3. Citation audit reconciliation ----

citation_summary = {
    "query_count": int(len(citation_audit)),
    "citation_present_count": int(
        citation_audit["citation_present"].astype(bool).sum()
    ),
    "normalized_format_pass_count": int(
        citation_audit["citation_format_pass"].astype(bool).sum()
    ),
    "invalid_label_occurrences": int(
        citation_audit["invalid_citation_labels"].apply(
            lambda x: len(x) if isinstance(x, (list, tuple, set)) else 0
        ).sum()
    ),
    "parser_result_changed_count": int(
        citation_audit["parser_result_changed"].astype(bool).sum()
    ),
}

print("\nCITATION AUDIT RECOMPUTED SUMMARY")
print(json.dumps(citation_summary, indent=2, default=str))

# ---- 4. Explicit metric-definition guardrail ----

print("\nMETRIC DEFINITION GUARDRAIL")
print("- N3 headline MRR is direct_mrr_at_10, not useful_mrr_at_10.")
print("- Useful MRR and direct MRR use different relevance definitions.")
print("- Recall is pool-relative to the union of the three top-10 result sets.")
print("- N4 normalized citation-format results are not the same as strict ASCII compliance.")

# Keep outputs in memory only.
PART3C_RETRIEVAL_RECONCILIATION = retrieval_reconciliation
PART3C_GENERATION_RECONCILIATION = n4_reconciliation
PART3C_CITATION_SUMMARY = citation_summary

print("\nPart 3C executed. No files written.")


RETRIEVAL METRIC RECONCILIATION


,source,method,metric,recomputed,frozen_report,absolute_difference,status
0,N3,BM25,direct_mrr_at_10,0.494444,0.494444,0.0,MATCH
1,N3,Dense,direct_mrr_at_10,0.451025,0.451025,0.0,MATCH
2,N3,Hybrid RRF,direct_mrr_at_10,0.673611,0.673611,0.0,MATCH
3,N3,BM25,graded_ndcg_at_10,0.511835,0.511835,0.0,MATCH
4,N3,Dense,graded_ndcg_at_10,0.582095,0.582095,0.0,MATCH
5,N3,Hybrid RRF,graded_ndcg_at_10,0.598620,0.598620,0.0,MATCH
6,N3,BM25,useful_mrr_at_10,0.690972,NaN,NaN,NOT_IN_FROZEN_HEADLINE
7,N3,Dense,useful_mrr_at_10,0.681349,NaN,NaN,NOT_IN_FROZEN_HEADLINE
8,N3,Hybrid RRF,useful_mrr_at_10,0.750000,NaN,NaN,NOT_IN_FROZEN_HEADLINE
9,N3,BM25,useful_pool_recall_at_10,0.568575,0.568575,0.0,MATCH



N4 HEADLINE RESULTS STRUCTURE
Type: dict
Keys: ['generation_completion_count', 'generation_completion_rate', 'structured_response_validity_count', 'structured_response_validity_rate', 'strict_ascii_citation_compliance_count', 'strict_ascii_citation_compliance_rate', 'unicode_normalized_citation_presence_count', 'unicode_normalized_citation_presence_rate', 'invalid_citation_label_occurrences', 'answer_correctness_score_rate', 'evidence_support_score_rate', 'answer_completeness_score_rate', 'citation_correctness_score_rate', 'overall_manual_rubric_score', 'overall_manual_rubric_maximum', 'overall_manual_rubric_score_rate', 'fully_passing_answer_count', 'fully_passing_answer_rate', 'acceptable_answer_count', 'acceptable_answer_rate', 'material_failure_count', 'material_failure_rate']

GENERATION METRIC RECONCILIATION


,metric,recomputed,frozen_report,status
0,answer_correctness_score_rate,0.875000,0.875000,MATCH
1,evidence_support_score_rate,0.958333,0.958333,MATCH
2,answer_completeness_score_rate,0.875000,0.875000,MATCH
3,citation_correctness_score_rate,0.958333,0.958333,MATCH
4,overall_manual_rubric_score_rate,0.916667,0.916667,MATCH
5,acceptable_answer_rate,0.916667,0.916667,MATCH
6,fully_passing_answer_rate,0.750000,0.750000,MATCH
7,material_failure_rate,0.083333,0.083333,MATCH



CITATION AUDIT RECOMPUTED SUMMARY
{
  "query_count": 12,
  "citation_present_count": 12,
  "normalized_format_pass_count": 12,
  "invalid_label_occurrences": 0,
  "parser_result_changed_count": 4
}

METRIC DEFINITION GUARDRAIL
- N3 headline MRR is direct_mrr_at_10, not useful_mrr_at_10.
- Useful MRR and direct MRR use different relevance definitions.
- Recall is pool-relative to the union of the three top-10 result sets.
- N4 normalized citation-format results are not the same as strict ASCII compliance.

Part 3C executed. No files written.


Query-level failure analysis

In [ ]:
print("PART2_FRAMES keys:")
print(sorted(PART2_FRAMES.keys()))

print("\nN3_INPUTS keys and artifact types:")
for key, value in N3_INPUTS.items():
    print(f"{key}: {type(value).__name__} — {value}")

print("\nPART2_FRAMES object types:")
for key, value in PART2_FRAMES.items():
    shape = getattr(value, "shape", None)
    print(f"{key}: {type(value).__name__}, shape={shape}")

PART2_FRAMES keys:
['citation_audit', 'generation_scores', 'human_judgments', 'per_query_metrics', 'qrels', 'queries', 'retrieval_runs']

N3_INPUTS keys and artifact types:
retrieval_summary: PosixPath — /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_summary_v2.json
retrieval_config: PosixPath — /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_config_v2.json
retrieval_runs: PosixPath — /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_runs_v2.parquet
per_query_metrics: PosixPath — /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_per_query_metrics_v2.parquet
heldout_queries: PosixPath — /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_queries_v2.parquet
human_judgments: PosixPath — /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_human_judgments_final_v2.parquet
human_judgments_manifest: PosixPath — /content/drive/My

In [ ]:

import pandas as pd
import numpy as np

# Use the verified in-memory frames
queries = PART2_FRAMES["queries"].copy()
retrieval_metrics = PART2_FRAMES["per_query_metrics"].copy()
retrieval_runs = PART2_FRAMES["retrieval_runs"].copy()
judgments = PART2_FRAMES["human_judgments"].copy()
qrels = PART2_FRAMES["qrels"].copy()
scores = PART2_FRAMES["generation_scores"].copy()
citation_audit = PART2_FRAMES["citation_audit"].copy()

# 1. Retrieval performance by query and method
retrieval_cols = [
    "query_id", "method", "useful_mrr_at_10",
    "graded_ndcg_at_10", "useful_precision_at_10",
    "useful_pool_recall_at_10", "direct_mrr_at_10",
    "direct_precision_at_10", "direct_pool_recall_at_10",
]

available_retrieval_cols = [
    c for c in retrieval_cols if c in retrieval_metrics.columns
]
retrieval_view = retrieval_metrics[available_retrieval_cols].copy()

# 2. Join query metadata, generation scores, and citation audit
query_view = (
    queries
    .merge(scores, on="query_id", how="left", suffixes=("", "_generation"))
    .merge(
        citation_audit[
            [
                c for c in [
                    "query_id", "citation_present", "citation_format_pass",
                    "invalid_citation_labels", "parser_result_changed",
                    "selected_evidence_count",
                ]
                if c in citation_audit.columns
            ]
        ],
        on="query_id",
        how="left",
        suffixes=("", "_citation"),
    )
)

# 3. Flag weaker retrieval cases using explicit diagnostic thresholds.
# These are diagnostic flags, not formal pass/fail standards.
retrieval_view["weak_useful_mrr"] = (
    retrieval_view["useful_mrr_at_10"] < 0.50
    if "useful_mrr_at_10" in retrieval_view.columns
    else False
)
retrieval_view["weak_direct_mrr"] = (
    retrieval_view["direct_mrr_at_10"] < 0.50
    if "direct_mrr_at_10" in retrieval_view.columns
    else False
)
retrieval_view["weak_ndcg"] = (
    retrieval_view["graded_ndcg_at_10"] < 0.50
    if "graded_ndcg_at_10" in retrieval_view.columns
    else False
)

# 4. Identify generation-quality risks
if "total_score_rate" in query_view.columns:
    query_view["low_generation_score"] = query_view["total_score_rate"] < 0.875
if "material_failure" in query_view.columns:
    query_view["material_failure"] = (
        query_view["material_failure"].fillna(False).astype(bool)
    )
if "acceptable_at_least_7_of_8" in query_view.columns:
    query_view["not_acceptable"] = ~(
        query_view["acceptable_at_least_7_of_8"].fillna(False).astype(bool)
    )

print("=== QUERY-LEVEL RETRIEVAL METRICS ===")
display(
    retrieval_view.sort_values(["query_id", "method"]).reset_index(drop=True)
)

print("\n=== GENERATION / CITATION RISK OVERVIEW ===")
risk_cols = [
    c for c in [
        "query_id", "ticker", "question_type", "difficulty", "query",
        "total_score_rate", "acceptable_at_least_7_of_8",
        "material_failure", "failure_category", "review_notes",
        "citation_present", "citation_format_pass",
        "invalid_citation_labels", "parser_result_changed",
        "selected_evidence_count",
    ]
    if c in query_view.columns
]
display(
    query_view[risk_cols]
    .sort_values("query_id")
    .reset_index(drop=True)
)

print("\n=== RETRIEVAL WEAKNESS COUNTS ===")
for col in ["weak_useful_mrr", "weak_direct_mrr", "weak_ndcg"]:
    if col in retrieval_view.columns:
        print(f"{col}: {int(retrieval_view[col].sum())} of {len(retrieval_view)} query-method rows")

print("\n=== GENERATION RISK COUNTS ===")
for col in ["low_generation_score", "material_failure", "not_acceptable"]:
    if col in query_view.columns:
        print(f"{col}: {int(query_view[col].sum())} of {len(query_view)} queries")

print("\nNo files written; no LLM calls made.")


=== QUERY-LEVEL RETRIEVAL METRICS ===


,query_id,method,useful_mrr_at_10,graded_ndcg_at_10,useful_precision_at_10,useful_pool_recall_at_10,direct_mrr_at_10,direct_precision_at_10,direct_pool_recall_at_10,weak_useful_mrr,weak_direct_mrr,weak_ndcg
0,H01,BM25,1.000000,0.548562,0.7,0.636364,0.333333,0.3,0.600000,False,True,False
1,H01,Dense,1.000000,0.572269,0.6,0.545455,0.500000,0.3,0.600000,False,False,False
2,H01,Hybrid RRF,1.000000,0.611385,0.6,0.545455,1.000000,0.3,0.600000,False,False,False
3,H02,BM25,1.000000,0.373725,0.6,0.461538,0.166667,0.1,0.200000,False,True,True
4,H02,Dense,1.000000,0.812349,0.8,0.615385,0.500000,0.5,1.000000,False,False,False
5,H02,Hybrid RRF,1.000000,0.897061,0.9,0.692308,1.000000,0.4,0.800000,False,False,False
6,H03,BM25,0.125000,0.180600,0.3,0.375000,0.100000,0.1,0.333333,True,True,True
7,H03,Dense,1.000000,0.768352,0.8,1.000000,0.333333,0.3,1.000000,False,True,False
8,H03,Hybrid RRF,0.500000,0.463745,0.5,0.625000,0.250000,0.2,0.666667,False,True,True
9,H04,BM25,1.000000,0.698630,0.6,0.666667,1.000000,0.1,0.500000,False,False,False



=== GENERATION / CITATION RISK OVERVIEW ===


,query_id,ticker,question_type,difficulty,query,total_score_rate,acceptable_at_least_7_of_8,material_failure,failure_category,review_notes,citation_present,citation_format_pass,invalid_citation_labels,parser_result_changed,selected_evidence_count
0,H01,AAPL,reported_driver,medium,"According to Apple's 2025 Form 10-K, what fact...",0.875,True,False,incomplete_answer,The answer accurately reports the retrieved Ma...,True,True,[],False,7
1,H02,AAPL,geographic_or_segment_comparison,medium,"According to Apple's 2024 Form 10-K, how did n...",1.000,True,False,none,Correctly reports the change and 2024 net-sale...,True,True,[],True,8
2,H03,MSFT,segment_driver,medium,"According to Microsoft's 2025 Form 10-K, what ...",1.000,True,False,none,Correctly identifies Azure as the principal In...,True,True,[],False,7
3,H04,MSFT,capital_expenditure,medium,"According to Microsoft's 2024 Form 10-K, what ...",0.875,True,False,minor_claim_conflation,"The facilities, datacenters, computer systems,...",True,True,[],False,7
4,H05,AMZN,segment_comparison,medium,"According to Amazon's 2024 Form 10-K, how did ...",1.000,True,False,none,Correctly reports AWS net sales and operating ...,True,True,[],False,7
5,H06,AMZN,cash_flow_driver,hard,"According to Amazon's 2023 Form 10-K, what fac...",1.000,True,False,none,Correctly identifies improved net income after...,True,True,[],True,7
6,H07,GOOGL,revenue_driver,medium,"According to Alphabet's 2024 Form 10-K, what f...",1.000,True,False,none,"Correctly distinguishes the drivers of Search,...",True,True,[],False,8
7,H08,GOOGL,risk_factor,hard,"According to Alphabet's 2023 Form 10-K, what r...",1.000,True,False,none,"Provides a broad, evidence-supported account o...",True,True,[],True,7
8,H09,NVDA,segment_driver,medium,"According to NVIDIA's 2025 Form 10-K, what fac...",1.000,True,False,none,Correctly attributes Data Center growth to Hop...,True,True,[],False,7
9,H10,NVDA,supply_risk,hard,"According to NVIDIA's 2024 Form 10-K, what sup...",1.000,True,False,none,Comprehensively and accurately summarizes foun...,True,True,[],False,7



=== RETRIEVAL WEAKNESS COUNTS ===
weak_useful_mrr: 8 of 36 query-method rows
weak_direct_mrr: 15 of 36 query-method rows
weak_ndcg: 13 of 36 query-method rows

=== GENERATION RISK COUNTS ===
low_generation_score: 1 of 12 queries
material_failure: 1 of 12 queries
not_acceptable: 1 of 12 queries

No files written; no LLM calls made.


In [ ]:

# Part 4B — H12 end-to-end failure trace
# Read-only diagnostic: no LLM calls, no artifact writes.

import json
import pandas as pd
from pathlib import Path

TARGET_QUERY_ID = "H12"

# Use already-loaded, verified dataframes
queries = PART2_FRAMES["queries"].copy()
retrieval_runs = PART2_FRAMES["retrieval_runs"].copy()
judgments = PART2_FRAMES["human_judgments"].copy()
qrels = PART2_FRAMES["qrels"].copy()
scores = PART2_FRAMES["generation_scores"].copy()

# Load the existing frozen trace file
with open(N4_INPUTS["generation_traces"], "r", encoding="utf-8") as f:
    trace_data = json.load(f)

print("Trace container type:", type(trace_data).__name__)
if isinstance(trace_data, dict):
    print("Top-level trace keys:", list(trace_data.keys())[:30])
    traces = trace_data.get("traces", trace_data.get("generations", trace_data))
elif isinstance(trace_data, list):
    traces = trace_data
else:
    raise TypeError(f"Unexpected trace container: {type(trace_data)}")

if isinstance(traces, dict):
    traces = list(traces.values())

h12_query = queries.loc[queries["query_id"].astype(str) == TARGET_QUERY_ID]
if h12_query.empty:
    raise ValueError(f"{TARGET_QUERY_ID} not found in query metadata")

print("\n" + "=" * 90)
print("1. H12 QUERY METADATA")
print("=" * 90)
display(h12_query.T)

h12_score = scores.loc[scores["query_id"].astype(str) == TARGET_QUERY_ID]
print("\n" + "=" * 90)
print("2. H12 GENERATION REVIEW")
print("=" * 90)
display(h12_score.T)

# Find H12 trace without assuming its exact nested schema
h12_trace_matches = [
    item for item in traces
    if isinstance(item, dict)
    and str(item.get("query_id", item.get("id", ""))) == TARGET_QUERY_ID
]

print("\n" + "=" * 90)
print("3. H12 TRACE STRUCTURE")
print("=" * 90)
print("Matching trace records:", len(h12_trace_matches))

if h12_trace_matches:
    h12_trace = h12_trace_matches[0]
    print("Trace fields:", list(h12_trace.keys()))
    for key, value in h12_trace.items():
        if isinstance(value, (dict, list)):
            print(f"\n{key}: type={type(value).__name__}, length={len(value)}")
            if isinstance(value, list) and value and isinstance(value[0], dict):
                print("  first item keys:", list(value[0].keys()))
            elif isinstance(value, dict):
                print("  nested keys:", list(value.keys())[:40])
        else:
            text_value = str(value)
            print(f"\n{key}: {text_value[:1500]}")
else:
    h12_trace = None
    print("Could not find H12 by a top-level query_id/id field.")
    print("Inspect the trace structure printed above before proceeding.")

# Inspect H12 retrieval records and show likely rank/evidence columns
h12_runs = retrieval_runs.loc[
    retrieval_runs["query_id"].astype(str) == TARGET_QUERY_ID
].copy()

print("\n" + "=" * 90)
print("4. H12 RETRIEVAL RUN SCHEMA")
print("=" * 90)
print("Shape:", h12_runs.shape)
print("Columns:", h12_runs.columns.tolist())

# Display metadata for each retrieval method, including any obvious rank fields.
rank_candidates = [
    c for c in h12_runs.columns
    if any(term in c.lower() for term in ["rank", "score", "chunk", "text", "content", "document", "evidence", "row", "ticker", "filing"])
]
id_candidates = [
    c for c in ["query_id", "retrieval_method", "evaluation_method", "method"]
    if c in h12_runs.columns
]
show_cols = list(dict.fromkeys(id_candidates + rank_candidates))
display(h12_runs[show_cols].head(40))

# Inspect H12 human judgments and qrels
print("\n" + "=" * 90)
print("5. H12 HUMAN JUDGMENTS")
print("=" * 90)
h12_judgments = judgments.loc[
    judgments["query_id"].astype(str) == TARGET_QUERY_ID
].copy()
print("Shape:", h12_judgments.shape)
display(h12_judgments)

print("\n" + "=" * 90)
print("6. H12 QRELS")
print("=" * 90)
h12_qrels = qrels.loc[
    qrels["query_id"].astype(str) == TARGET_QUERY_ID
].copy()
print("Shape:", h12_qrels.shape)
display(h12_qrels)

print("\n" + "=" * 90)
print("7. RELEVANCE LABEL DISTRIBUTION")
print("=" * 90)
for name, frame, label_col in [
    ("Human judgments", h12_judgments, "human_relevance"),
    ("Qrels", h12_qrels, "relevance_grade"),
]:
    if label_col in frame.columns:
        print(f"\n{name}:")
        print(frame[label_col].value_counts(dropna=False).sort_index())

print("\nDiagnostic complete. No files written; no LLM calls made.")


Trace container type: list

1. H12 QUERY METADATA


,11
query_id,H12
ticker,MSFT
report_year,2023
question_type,risk_factor
difficulty,hard
query,"According to Microsoft's 2023 Form 10-K, what ..."
sec_filing_year,2023
filing_date,2023-07-27
report_date,2023-06-30
document_id,msft_10k_000095017023035122



2. H12 GENERATION REVIEW


,11
query_id,H12
query_text,"According to Microsoft's 2023 Form 10-K, what ..."
answer,Microsoft warned that failures in its operatio...
abstained,False
answer_correctness_score_0_to_2,0
evidence_support_score_0_to_2,1
answer_completeness_score_0_to_2,0
citation_correctness_score_0_to_2,1
total_score_0_to_8,2
total_score_rate,0.25



3. H12 TRACE STRUCTURE
Matching trace records: 1
Trace fields: ['query_id', 'status', 'generation_policy_version', 'context_policy_version', 'retrieval_method', 'corpus_version', 'provider', 'model', 'structured_output_enabled', 'sampling_parameters_supplied', 'max_output_tokens', 'attempt_number', 'started_at_utc', 'latency_seconds', 'query_text', 'context_sha256', 'context_characters', 'retrieved_evidence_count', 'selected_evidence_count', 'available_citation_labels', 'generation', 'citation_validation', 'usage', 'raw_response']

query_id: H12

status: COMPLETED

generation_policy_version: groq_gpt_oss_120b_v2

context_policy_version: v1

retrieval_method: hybrid_rrf

corpus_version: sec30_959ac4a68dbf

provider: groq

model: groq/openai/gpt-oss-120b

structured_output_enabled: True

sampling_parameters_supplied: False

max_output_tokens: 1536

attempt_number: 1

started_at_utc: 2026-08-10T10:24:38.183582+00:00

latency_seconds: 1.4738

query_text: According to Microsoft's 2023 Form

,query_id,retrieval_method,evaluation_method,rank,retrieval_score,bm25_rank,dense_rank,corpus_row,chunk_id,document_id,chunk_index,text,ticker,filing_date,primary_document,raw_content_sha256,processed_content_sha256,query_ticker,query_filing_date
330,H12,dense_filtered,dense,1,0.712218,<NA>,<NA>,2523,msft_10k_000095017023035122__chunk_0176,msft_10k_000095017023035122,176,"Long-lived assets, excluding financial instrum...",MSFT,2023-07-27,msft-20230630.htm,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,MSFT,2023-07-27
331,H12,dense_filtered,dense,2,0.708774,<NA>,<NA>,2539,msft_10k_000095017023035122__chunk_0192,msft_10k_000095017023035122,192,"and The Bank of New York Mellon Trust Company,...",MSFT,2023-07-27,msft-20230630.htm,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,MSFT,2023-07-27
332,H12,dense_filtered,dense,3,0.704512,<NA>,<NA>,2425,msft_10k_000095017023035122__chunk_0078,msft_10k_000095017023035122,78,Our Board of Directors declared the following ...,MSFT,2023-07-27,msft-20230630.htm,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,MSFT,2023-07-27
333,H12,dense_filtered,dense,4,0.703185,<NA>,<NA>,2347,msft_10k_000095017023035122__chunk_0000,msft_10k_000095017023035122,0,10-K UNITED STATES SECURITIES AND EXCHANGE COM...,MSFT,2023-07-27,msft-20230630.htm,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,MSFT,2023-07-27
334,H12,dense_filtered,dense,5,0.698241,<NA>,<NA>,2351,msft_10k_000095017023035122__chunk_0004,msft_10k_000095017023035122,4,These forward-looking statements generally are...,MSFT,2023-07-27,msft-20230630.htm,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,MSFT,2023-07-27
335,H12,dense_filtered,dense,6,0.697655,<NA>,<NA>,2426,msft_10k_000095017023035122__chunk_0079,msft_10k_000095017023035122,79,OVERVIEW Microsoft is a technology company who...,MSFT,2023-07-27,msft-20230630.htm,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,MSFT,2023-07-27
336,H12,dense_filtered,dense,7,0.696734,<NA>,<NA>,2402,msft_10k_000095017023035122__chunk_0055,msft_10k_000095017023035122,55,"At times, to achieve commercial objectives, we...",MSFT,2023-07-27,msft-20230630.htm,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,MSFT,2023-07-27
337,H12,dense_filtered,dense,8,0.691760,<NA>,<NA>,2394,msft_10k_000095017023035122__chunk_0047,msft_10k_000095017023035122,47,These include unauthorized use of accounts thr...,MSFT,2023-07-27,msft-20230630.htm,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,MSFT,2023-07-27
338,H12,dense_filtered,dense,9,0.688025,<NA>,<NA>,2536,msft_10k_000095017023035122__chunk_0189,msft_10k_000095017023035122,189,"due 2043, dated as of December 6, 2013, betwee...",MSFT,2023-07-27,msft-20230630.htm,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,MSFT,2023-07-27
339,H12,dense_filtered,dense,10,0.686068,<NA>,<NA>,2403,msft_10k_000095017023035122__chunk_0056,msft_10k_000095017023035122,56,Despite our efforts to protect customer and us...,MSFT,2023-07-27,msft-20230630.htm,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,31bc9a326f3daa04ca1aee5ed5120e5ab40cde0a0ba604...,MSFT,2023-07-27



5. H12 HUMAN JUDGMENTS
Shape: (20, 27)


,query_id,corpus_row,chunk_id,document_id,chunk_index,text,ticker,company,form_type,filing_date,...,dense_score,hybrid_score,query,question_type,difficulty,report_year,retrieved_by_count,review_order,human_relevance,review_reason
194,H12,2406,msft_10k_000095017023035122__chunk_0059,msft_10k_000095017023035122,59,Some AI scenarios present ethical issues or ma...,MSFT,MICROSOFT CORP,10-K,2023-07-27,...,NaN,0.044089,"According to Microsoft's 2023 Form 10-K, what ...",risk_factor,hard,2023,1,1,0,
195,H12,2403,msft_10k_000095017023035122__chunk_0056,msft_10k_000095017023035122,56,Despite our efforts to protect customer and us...,MSFT,MICROSOFT CORP,10-K,2023-07-27,...,0.686068,NaN,"According to Microsoft's 2023 Form 10-K, what ...",risk_factor,hard,2023,1,2,1,
196,H12,2396,msft_10k_000095017023035122__chunk_0049,msft_10k_000095017023035122,49,It may take longer than expected to realize th...,MSFT,MICROSOFT CORP,10-K,2023-07-27,...,NaN,NaN,"According to Microsoft's 2023 Form 10-K, what ...",risk_factor,hard,2023,1,3,2,
197,H12,2394,msft_10k_000095017023035122__chunk_0047,msft_10k_000095017023035122,47,These include unauthorized use of accounts thr...,MSFT,MICROSOFT CORP,10-K,2023-07-27,...,0.691760,NaN,"According to Microsoft's 2023 Form 10-K, what ...",risk_factor,hard,2023,1,4,1,
198,H12,2347,msft_10k_000095017023035122__chunk_0000,msft_10k_000095017023035122,0,10-K UNITED STATES SECURITIES AND EXCHANGE COM...,MSFT,MICROSOFT CORP,10-K,2023-07-27,...,0.703185,NaN,"According to Microsoft's 2023 Form 10-K, what ...",risk_factor,hard,2023,1,5,0,
199,H12,2532,msft_10k_000095017023035122__chunk_0185,msft_10k_000095017023035122,185,"Also, projections of any evaluation of effecti...",MSFT,MICROSOFT CORP,10-K,2023-07-27,...,NaN,0.045546,"According to Microsoft's 2023 Form 10-K, what ...",risk_factor,hard,2023,2,6,0,
200,H12,2430,msft_10k_000095017023035122__chunk_0083,msft_10k_000095017023035122,83,Additional information on our reportable segme...,MSFT,MICROSOFT CORP,10-K,2023-07-27,...,NaN,NaN,"According to Microsoft's 2023 Form 10-K, what ...",risk_factor,hard,2023,1,7,0,
201,H12,2402,msft_10k_000095017023035122__chunk_0055,msft_10k_000095017023035122,55,"At times, to achieve commercial objectives, we...",MSFT,MICROSOFT CORP,10-K,2023-07-27,...,0.696734,NaN,"According to Microsoft's 2023 Form 10-K, what ...",risk_factor,hard,2023,1,8,2,
202,H12,2523,msft_10k_000095017023035122__chunk_0176,msft_10k_000095017023035122,176,"Long-lived assets, excluding financial instrum...",MSFT,MICROSOFT CORP,10-K,2023-07-27,...,0.712218,0.032258,"According to Microsoft's 2023 Form 10-K, what ...",risk_factor,hard,2023,2,9,0,
203,H12,2457,msft_10k_000095017023035122__chunk_0110,msft_10k_000095017023035122,110,The tax benefits recognized in the financial s...,MSFT,MICROSOFT CORP,10-K,2023-07-27,...,NaN,NaN,"According to Microsoft's 2023 Form 10-K, what ...",risk_factor,hard,2023,1,10,0,



6. H12 QRELS
Shape: (20, 8)


,query_id,corpus_row,chunk_id,relevance_grade,adjudication_note,label_source,human_reviewed,rubric_version
194,H12,2347,msft_10k_000095017023035122__chunk_0000,0,,human,True,heldout_relevance_v2_0_1_2
195,H12,2351,msft_10k_000095017023035122__chunk_0004,0,,human,True,heldout_relevance_v2_0_1_2
196,H12,2352,msft_10k_000095017023035122__chunk_0005,0,,human,True,heldout_relevance_v2_0_1_2
197,H12,2361,msft_10k_000095017023035122__chunk_0014,0,,human,True,heldout_relevance_v2_0_1_2
198,H12,2394,msft_10k_000095017023035122__chunk_0047,1,,human,True,heldout_relevance_v2_0_1_2
199,H12,2396,msft_10k_000095017023035122__chunk_0049,2,,human,True,heldout_relevance_v2_0_1_2
200,H12,2402,msft_10k_000095017023035122__chunk_0055,2,,human,True,heldout_relevance_v2_0_1_2
201,H12,2403,msft_10k_000095017023035122__chunk_0056,1,,human,True,heldout_relevance_v2_0_1_2
202,H12,2406,msft_10k_000095017023035122__chunk_0059,0,,human,True,heldout_relevance_v2_0_1_2
203,H12,2425,msft_10k_000095017023035122__chunk_0078,0,,human,True,heldout_relevance_v2_0_1_2



7. RELEVANCE LABEL DISTRIBUTION

Human judgments:
human_relevance
0    16
1     2
2     2
Name: count, dtype: Int64

Qrels:
relevance_grade
0    16
1     2
2     2
Name: count, dtype: Int64

Diagnostic complete. No files written; no LLM calls made.


In [ ]:

# Part 4B.2 — Map selected H12 evidence to the reviewed relevance labels
# Read-only; uses the existing in-memory trace and dataframes.

import pandas as pd

assert h12_trace is not None, "H12 trace was not loaded in Part 4B."

print("Top-level trace fields:", list(h12_trace.keys()))

# Print nested generation and citation-validation structures.
for key in ["generation", "citation_validation"]:
    print(f"\n--- {key} ---")
    print(h12_trace.get(key))

# Available citation labels can map S1, S2, ... to chunk metadata.
labels = h12_trace.get("available_citation_labels", [])
print("\n--- Available citation labels ---")
print("Type:", type(labels).__name__)
print(labels)

# Display all H12 judged chunks with readable text and labels.
text_col = "text"
judgment_cols = [
    c for c in [
        "corpus_row", "chunk_id", "chunk_index",
        "human_relevance", "retrieved_by_count",
        "bm25_rank", "dense_rank", "hybrid_rank",
        "text",
    ]
    if c in h12_judgments.columns
]

print("\n--- Human-reviewed H12 chunks ---")
display(h12_judgments[judgment_cols].sort_values(
    "human_relevance", ascending=False
).reset_index(drop=True))

# Display hybrid top 10 with full chunk text.
hybrid_h12 = h12_runs[
    h12_runs["retrieval_method"].astype(str)
    == "hybrid_rrf_asymmetric_filtered"
].sort_values("rank")

print("\n--- Hybrid RRF top 10: full text ---")
hybrid_cols = [
    c for c in [
        "rank", "retrieval_score", "chunk_id",
        "bm25_rank", "dense_rank", "text",
    ]
    if c in hybrid_h12.columns
]
display(hybrid_h12[hybrid_cols].reset_index(drop=True))

print("\nNo files written; no LLM calls made.")


Top-level trace fields: ['query_id', 'status', 'generation_policy_version', 'context_policy_version', 'retrieval_method', 'corpus_version', 'provider', 'model', 'structured_output_enabled', 'sampling_parameters_supplied', 'max_output_tokens', 'attempt_number', 'started_at_utc', 'latency_seconds', 'query_text', 'context_sha256', 'context_characters', 'retrieved_evidence_count', 'selected_evidence_count', 'available_citation_labels', 'generation', 'citation_validation', 'usage', 'raw_response']

--- generation ---
{'answer': 'Microsoft warned that failures in its operational and security controls could lead to excessive outages, data loss and disruptions of its online services.  Such incidents could damage its brand, hurt its reputation and adversely affect its business and consolidated financial statements.  The company also noted that unintended or controversial AI applications could cause brand or reputational harm, which likewise could negatively impact its business and financial res

,corpus_row,chunk_id,chunk_index,human_relevance,retrieved_by_count,bm25_rank,dense_rank,hybrid_rank,text
0,2396,msft_10k_000095017023035122__chunk_0049,49,2,1,6,<NA>,<NA>,It may take longer than expected to realize th...
1,2402,msft_10k_000095017023035122__chunk_0055,55,2,1,<NA>,7,<NA>,"At times, to achieve commercial objectives, we..."
2,2394,msft_10k_000095017023035122__chunk_0047,47,1,1,<NA>,8,<NA>,These include unauthorized use of accounts thr...
3,2403,msft_10k_000095017023035122__chunk_0056,56,1,1,<NA>,10,<NA>,Despite our efforts to protect customer and us...
4,2406,msft_10k_000095017023035122__chunk_0059,59,0,1,<NA>,<NA>,6,Some AI scenarios present ethical issues or ma...
5,2347,msft_10k_000095017023035122__chunk_0000,0,0,1,<NA>,4,<NA>,10-K UNITED STATES SECURITIES AND EXCHANGE COM...
6,2532,msft_10k_000095017023035122__chunk_0185,185,0,2,7,<NA>,4,"Also, projections of any evaluation of effecti..."
7,2430,msft_10k_000095017023035122__chunk_0083,83,0,1,3,<NA>,<NA>,Additional information on our reportable segme...
8,2523,msft_10k_000095017023035122__chunk_0176,176,0,2,<NA>,1,8,"Long-lived assets, excluding financial instrum..."
9,2457,msft_10k_000095017023035122__chunk_0110,110,0,1,9,<NA>,<NA>,The tax benefits recognized in the financial s...



--- Hybrid RRF top 10: full text ---


,rank,retrieval_score,chunk_id,bm25_rank,dense_rank,text
0,1,0.059715,msft_10k_000095017023035122__chunk_0078,4,3,Our Board of Directors declared the following ...
1,2,0.052962,msft_10k_000095017023035122__chunk_0004,11,5,These forward-looking statements generally are...
2,3,0.052778,msft_10k_000095017023035122__chunk_0079,10,6,OVERVIEW Microsoft is a technology company who...
3,4,0.045546,msft_10k_000095017023035122__chunk_0185,7,24,"Also, projections of any evaluation of effecti..."
4,5,0.044798,msft_10k_000095017023035122__chunk_0191,19,11,"Bank National Association, as Trustee, to the ..."
5,6,0.044089,msft_10k_000095017023035122__chunk_0059,13,18,Some AI scenarios present ethical issues or ma...
6,7,0.032258,msft_10k_000095017023035122__chunk_0082,1,<NA>,"Third, we are consolidating our leases to crea..."
7,8,0.032258,msft_10k_000095017023035122__chunk_0176,<NA>,1,"Long-lived assets, excluding financial instrum..."
8,9,0.031250,msft_10k_000095017023035122__chunk_0014,2,<NA>,We are 28 percent of the way for Hispanic and ...
9,10,0.031250,msft_10k_000095017023035122__chunk_0192,<NA>,2,"and The Bank of New York Mellon Trust Company,..."



No files written; no LLM calls made.


In [ ]:
import json
from pathlib import Path

trace_path = Path(N4_INPUTS["generation_traces"])
with trace_path.open("r", encoding="utf-8") as f:
    trace_data = json.load(f)

if isinstance(trace_data, dict):
    records = trace_data.get("traces", trace_data.get("records", [trace_data]))
else:
    records = trace_data

h12 = next(r for r in records if r.get("query_id") == "H12")

def inspect_structure(obj, path="root", depth=0):
    indent = "  " * depth

    if isinstance(obj, dict):
        print(f"{indent}{path}: dict keys={list(obj.keys())}")
        if depth < 4:
            for k, v in obj.items():
                if isinstance(v, (dict, list)):
                    inspect_structure(v, f"{path}.{k}", depth + 1)
                elif any(term in k.lower() for term in
                         ["evidence", "context", "source", "citation", "chunk", "label"]):
                    print(f"{indent}  {k}: {str(v)[:500]}")

    elif isinstance(obj, list):
        print(f"{indent}{path}: list length={len(obj)}")
        if obj and depth < 4:
            inspect_structure(obj[0], f"{path}[0]", depth + 1)

inspect_structure(h12)

root: dict keys=['query_id', 'status', 'generation_policy_version', 'context_policy_version', 'retrieval_method', 'corpus_version', 'provider', 'model', 'structured_output_enabled', 'sampling_parameters_supplied', 'max_output_tokens', 'attempt_number', 'started_at_utc', 'latency_seconds', 'query_text', 'context_sha256', 'context_characters', 'retrieved_evidence_count', 'selected_evidence_count', 'available_citation_labels', 'generation', 'citation_validation', 'usage', 'raw_response']
  context_policy_version: v1
  context_sha256: a755ba4892b7bd1351c7955ffe9677085a464077855a3d222e2d5718069d7705
  context_characters: 16639
  retrieved_evidence_count: 10
  selected_evidence_count: 7
  root.available_citation_labels: list length=7
  root.generation: dict keys=['answer', 'abstained', 'abstention_reason']
  root.citation_validation: dict keys=['valid_labels', 'cited_labels', 'unique_cited_labels', 'invalid_labels', 'duplicate_citation_occurrences', 'validation_checks', 'overall_valid']
    

In [ ]:
import json
import re
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/04_grounded_answer_generation.ipynb"
)

nb = json.loads(notebook_path.read_text(encoding="utf-8"))

terms = re.compile(
    r"selected_evidence|available_citation_labels|context_policy|"
    r"citation_labels|citation_label|context_text|build_context|"
    r"retrieved_evidence|top_k|context_characters|\\[S",
    re.IGNORECASE,
)

for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    lines = "".join(cell.get("source", [])).splitlines()
    matching = [
        (j, line)
        for j, line in enumerate(lines, start=1)
        if terms.search(line)
    ]

    if matching:
        print(f"\n{'='*20} CELL {i} {'='*20}")
        # Show matching lines with nearby context.
        shown = set()
        for line_no, _ in matching:
            for j in range(max(1, line_no - 3), min(len(lines), line_no + 3) + 1):
                if j not in shown:
                    print(f"{j:>4}: {lines[j-1][:220]}")
                    shown.add(j)
            print("...")

PatternError: unterminated character set at position 163

In [ ]:
import json
import re
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/04_grounded_answer_generation.ipynb"
)

nb = json.loads(notebook_path.read_text(encoding="utf-8"))

terms = re.compile(
    r"selected_evidence|available_citation_labels|context_policy|"
    r"citation_labels|citation_label|context_text|build_context|"
    r"retrieved_evidence|top_k|context_characters|citation",
    re.IGNORECASE,
)

for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    lines = "".join(cell.get("source", [])).splitlines()
    matching_lines = [
        j for j, line in enumerate(lines)
        if terms.search(line)
    ]

    if matching_lines:
        print(f"\n{'=' * 16} CELL {i} {'=' * 16}")
        shown = set()

        for j in matching_lines:
            for k in range(max(0, j - 3), min(len(lines), j + 4)):
                if k not in shown:
                    print(f"{k + 1:>4}: {lines[k][:220]}")
                    shown.add(k)
            print("...")


================ CELL 1 ================
 336:     "dense_candidate_k": 30,
 337:     "bm25_candidate_k": 20,
 338:     "rrf_k": 30,
 339:     "final_top_k": 10,
 340:     "evaluation_pool_depth": 10,
 341:     "form_type": "10-K",
 342:     "expansion_applied": False,
...
 742:     dense_candidate_k: int
 743:     bm25_candidate_k: int
 744:     rrf_k: int
 745:     frozen_final_top_k: int
 746: 
 747:     allow_label_blind_dense_recovery: bool
 748:     allow_adjacent_chunk_expansion: bool
...
 750:     allow_evidence_diversification: bool
 751:     allow_evidence_adequacy_gating: bool
 752:     allow_answer_abstention: bool
 753:     require_citation_validation: bool
 754: 
 755:     allow_n3_artifact_overwrite: bool
 756:     allow_generation_label_access: bool
...
 777:     rrf_k=int(
 778:         n3_config["rrf_k"]
 779:     ),
 780:     frozen_final_top_k=int(
 781:         n3_config["final_top_k"]
 782:     ),
 783: 
...
 784:     allow_label_blind_dense_recovery=True,
...
 7

In [ ]:
from pathlib import Path
import pandas as pd
import json

artifact_root = Path(
    "/content/drive/MyDrive/FinanceRAG-Analyst/artifacts"
)

matches = list(
    artifact_root.rglob("notebook_04_control_context_audit_v1.parquet")
)

if not matches:
    print("Context audit artifact not found.")
    print("Artifact root exists:", artifact_root.exists())
    print("Search root:", artifact_root)
else:
    audit_path = matches[0]
    print("Audit path:", audit_path)

    audit = pd.read_parquet(audit_path)
    print("Columns:", audit.columns.tolist())

    h12 = audit.loc[
        audit["query_id"].astype(str).eq("H12")
    ]

    if h12.empty:
        print("H12 was not found in the context audit.")
    else:
        row = h12.iloc[0]

        for column in [
            "query_id",
            "retrieved_chunks",
            "selected_chunks",
            "skipped_chunks",
            "context_characters",
            "decision",
            "failed_checks",
            "selected_evidence",
            "skipped_evidence",
        ]:
            if column in audit.columns:
                value = row[column]
                print(f"\n--- {column} ---")

                if isinstance(value, (dict, list)):
                    print(json.dumps(value, indent=2, default=str))
                else:
                    print(value)

Audit path: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/manifests/notebook_04_control_context_audit_v1.parquet
Columns: ['query_id', 'retrieved_chunks', 'selected_chunks', 'skipped_chunks', 'context_characters', 'decision', 'failed_checks', 'context_sha256']

--- query_id ---
H12

--- retrieved_chunks ---
10

--- selected_chunks ---
7

--- skipped_chunks ---
3

--- context_characters ---
16639

--- decision ---
GENERATE

--- failed_checks ---



In [ ]:
from pathlib import Path
import json
import re

artifact_root = Path(
    "/content/drive/MyDrive/FinanceRAG-Analyst/artifacts"
)
notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/04_grounded_answer_generation.ipynb"
)

print("=== Related saved artifacts ===")
for path in sorted(artifact_root.rglob("*")):
    if path.is_file() and any(
        term in path.name.lower()
        for term in ["context", "evidence", "selected_chunk", "retrieval_trace"]
    ):
        print(path.relative_to(artifact_root))

print("\n=== Notebook 4 context construction source ===")
nb = json.loads(notebook_path.read_text(encoding="utf-8"))

terms = re.compile(
    r"control_context_audit|selected_chunks|skipped_chunks|"
    r"available_citation_labels|context_sha256|"
    r"context_parts|context_text|citation_label|"
    r"selected_evidence|retrieved_chunks",
    re.IGNORECASE,
)

for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    lines = "".join(cell.get("source", [])).splitlines()
    hits = [j for j, line in enumerate(lines) if terms.search(line)]

    if hits:
        print(f"\n--- CELL {i} ---")
        shown = set()
        for j in hits:
            for k in range(max(0, j - 4), min(len(lines), j + 5)):
                if k not in shown:
                    print(f"{k + 1:>4}: {lines[k][:240]}")
                    shown.add(k)

=== Related saved artifacts ===
generation/manifests/notebook_04_context_policy_manifest_v1.json
generation/manifests/notebook_04_control_context_audit_v1.parquet
generation/manifests/notebook_04_control_context_records_v1.json

=== Notebook 4 context construction source ===

--- CELL 7 ---
 478:     "results_per_query": EXPECTED_RESULTS_PER_QUERY,
 479:     "total_retrieval_rows": int(
 480:         len(frozen_control_evidence)
 481:     ),
 482:     "unique_retrieved_chunks": int(
 483:         frozen_control_evidence["chunk_id"].nunique()
 484:     ),
 485:     "validation": {
 486:         "heldout_query_coverage": "PASS",

--- CELL 8 ---
  29: # Character budgeting is deterministic and provider-independent.
  30: # It avoids tying the experiment to one provider's tokenizer.
  31: 
  32: MAX_CONTEXT_CHARACTERS = 18_000
  33: MAX_SELECTED_CHUNKS = 8
  34: MIN_SELECTED_CHUNKS = 2
  35: MIN_TOTAL_EVIDENCE_CHARACTERS = 800
  36: 
  37: # Require basic citation metadata for every select

In [ ]:
from pathlib import Path
import json

records_path = (
    Path("/content/drive/MyDrive/FinanceRAG-Analyst/artifacts")
    / "generation/manifests/notebook_04_control_context_records_v1.json"
)

with records_path.open("r", encoding="utf-8") as f:
    records = json.load(f)

print("JSON type:", type(records).__name__)

# Support either a list of records or a dictionary containing records.
if isinstance(records, dict):
    print("Top-level keys:", list(records.keys()))
    if "records" in records:
        records = records["records"]
    elif "context_records" in records:
        records = records["context_records"]
    else:
        records = [records] if "query_id" in records else []

h12_records = [
    r for r in records
    if str(r.get("query_id")) == "H12"
]

if not h12_records:
    print("H12 record not found. Inspect the JSON structure above.")
else:
    record = h12_records[0]
    print("Context SHA-256:", record.get("context_sha256"))
    print("Selected evidence count:", len(record.get("selected_evidence", [])))

    for evidence in record.get("selected_evidence", []):
        label = evidence.get("citation_label")
        print(f"\n{'=' * 12} {label} {'=' * 12}")
        for key in [
            "chunk_id",
            "retrieval_rank",
            "document_id",
            "ticker",
            "form_type",
            "filing_date",
            "text_sha256",
            "text",
            "evidence_text",
        ]:
            if key in evidence:
                value = evidence[key]
                print(f"{key}:")
                print(str(value)[:2500])

JSON type: list
Context SHA-256: a755ba4892b7bd1351c7955ffe9677085a464077855a3d222e2d5718069d7705
Selected evidence count: 7

============ S1 ============
chunk_id:
msft_10k_000095017023035122__chunk_0078
retrieval_rank:
1
document_id:
msft_10k_000095017023035122
ticker:
MSFT
form_type:
10-K
filing_date:
2023-07-27
text_sha256:
ed949a46560257b0fc78d3ec94be6ffcaa02f96cc43f8bf54b6900bcc2fd9cda
text:
Our Board of Directors declared the following dividends during the fourth quarter of fiscal year 2023: Declaration Date Record Date Payment Date Dividend Per Share Amount (In millions) June 13, 2023 August 17, 2023 September 14, 2023 $ 0.68 $ 5,054 We returned $9.7 billion to shareholders in the form of share repurchases and dividends in the fourth quarter of fiscal year 2023. Refer to Note 16 – Stockholders’ Equity of the Notes to Financial Statements (Part II, Item 8 of this Form 10-K) for further discussion regarding share repurchases and dividends. 38 PART II Item 6 ITEM 6. [R ESERVED] 39

In [ ]:
from pathlib import Path
import pandas as pd

repo_root = Path(
    "/content/drive/MyDrive/FinanceRAG-Analyst"
)

target_ids = {
    "msft_10k_000095017023035122__chunk_0049",
    "msft_10k_000095017023035122__chunk_0055",
}

print("Searching saved artifacts for relevant chunk records...")

for path in repo_root.rglob("*"):
    if not path.is_file():
        continue

    if path.suffix.lower() not in {".parquet", ".csv"}:
        continue

    try:
        if path.suffix.lower() == ".parquet":
            frame = pd.read_parquet(path)
        else:
            frame = pd.read_csv(path)
    except Exception:
        continue

    if "chunk_id" not in frame.columns:
        continue

    matches = frame.loc[
        frame["chunk_id"].astype(str).isin(target_ids)
    ]

    if not matches.empty:
        print(f"\nArtifact: {path}")
        print("Columns:", frame.columns.tolist())

        for _, row in matches.iterrows():
            print("\n" + "=" * 70)
            print("CHUNK:", row["chunk_id"])

            for col in [
                "ticker", "form_type", "filing_date",
                "section", "text", "chunk_text", "content"
            ]:
                if col in frame.columns and pd.notna(row[col]):
                    print(f"\n{col}:\n{str(row[col])[:5000]}")

Searching saved artifacts for relevant chunk records...

Artifact: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_runs_v2.parquet
Columns: ['rank', 'retrieval_method', 'retrieval_score', 'bm25_rank', 'dense_rank', 'corpus_row', 'chunk_id', 'document_id', 'chunk_index', 'start_span_index', 'end_span_index', 'start_char', 'end_char', 'token_count', 'below_minimum_target', 'text', 'ticker', 'company', 'cik', 'form_type', 'filing_date', 'report_date', 'accession_number', 'primary_document', 'source_url', 'raw_content_sha256', 'processed_content_sha256', 'processing_version', 'corpus_version', 'query_id', 'query', 'query_ticker', 'query_report_year', 'query_filing_date', 'evaluation_method']

CHUNK: msft_10k_000095017023035122__chunk_0055

ticker:
MSFT

form_type:
10-K

filing_date:
2023-07-27

text:
At times, to achieve commercial objectives, we may enter into agreements with larger liability exposure to customers. Our products operate in conjunction with 

In [ ]:

from pathlib import Path
import json
import pandas as pd

ROOT = Path("/content/drive/MyDrive/FinanceRAG-Analyst")
EVAL_DIR = ROOT / "artifacts" / "evaluation"

runs_path = EVAL_DIR / "heldout_retrieval_runs_v2.parquet"
config_path = EVAL_DIR / "heldout_retrieval_config_v2.json"

runs = pd.read_parquet(runs_path)
config = json.loads(config_path.read_text(encoding="utf-8"))

qid = "H12"
targets = {
    "msft_10k_000095017023035122__chunk_0049",
    "msft_10k_000095017023035122__chunk_0055",
}

h12 = runs.loc[runs["query_id"].eq(qid)].copy()

print("FROZEN CONFIG")
for key in [
    "dense_candidate_k",
    "bm25_candidate_k",
    "rrf_k",
    "final_top_k",
]:
    print(f"{key}: {config.get(key)}")

print("\nRELEVANT CHUNKS ACROSS SAVED RETRIEVAL RUNS")
cols = [
    "retrieval_method", "rank", "retrieval_score",
    "bm25_rank", "dense_rank", "chunk_id",
]
cols = [c for c in cols if c in h12.columns]

print(
    h12.loc[h12["chunk_id"].isin(targets), cols]
       .sort_values(["chunk_id", "retrieval_method"])
       .to_string(index=False)
)

print("\nTOP HYBRID RESULTS")
hybrid = h12.loc[
    h12["retrieval_method"].astype(str).str.contains(
        "hybrid", case=False, na=False
    )
].sort_values("rank")

print(
    hybrid[
        [c for c in [
            "rank", "retrieval_score", "bm25_rank",
            "dense_rank", "chunk_id", "text"
        ] if c in hybrid.columns]
    ].head(10).to_string(index=False, max_colwidth=180)
)

print("\nDIAGNOSTIC RRF CONTRIBUTIONS")
k = int(config["rrf_k"])

for chunk_id in sorted(targets):
    rows = h12.loc[h12["chunk_id"].eq(chunk_id)]
    if rows.empty:
        print(f"\n{chunk_id}: not found")
        continue

    # Use component ranks recorded in the saved artifacts.
    row = rows.iloc[0]
    bm25_rank = row.get("bm25_rank")
    dense_rank = row.get("dense_rank")

    def contribution(rank):
        return (
            1.0 / (k + float(rank))
            if pd.notna(rank) and float(rank) > 0
            else 0.0
        )

    bm25_contrib = contribution(bm25_rank)
    dense_contrib = contribution(dense_rank)

    print(f"\n{chunk_id}")
    print(f"  BM25 rank: {bm25_rank}")
    print(f"  Dense rank: {dense_rank}")
    print(f"  BM25 RRF contribution: {bm25_contrib:.6f}")
    print(f"  Dense RRF contribution: {dense_contrib:.6f}")
    print(f"  Sum of recorded component contributions: "
          f"{bm25_contrib + dense_contrib:.6f}")

print("\nRead-only diagnostic complete; no files written.")


FROZEN CONFIG
dense_candidate_k: 30
bm25_candidate_k: 20
rrf_k: 30
final_top_k: 10

RELEVANT CHUNKS ACROSS SAVED RETRIEVAL RUNS
retrieval_method  rank  retrieval_score  bm25_rank  dense_rank                                chunk_id
   bm25_filtered     6        16.895615       <NA>        <NA> msft_10k_000095017023035122__chunk_0049
  dense_filtered     7         0.696734       <NA>        <NA> msft_10k_000095017023035122__chunk_0055

TOP HYBRID RESULTS
 rank  retrieval_score  bm25_rank  dense_rank                                chunk_id                                                                                                                                                                                 text
    1         0.059715          4           3 msft_10k_000095017023035122__chunk_0078 Our Board of Directors declared the following dividends during the fourth quarter of fiscal year 2023: Declaration Date Record Date Payment Date Dividend Per Share Amount (In mi...
    2    

In [ ]:

from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/FinanceRAG-Analyst")
EVAL_DIR = ROOT / "artifacts" / "evaluation"

target_ids = {
    "msft_10k_000095017023035122__chunk_0049",
    "msft_10k_000095017023035122__chunk_0055",
}

candidate_paths = [
    EVAL_DIR / "heldout_judgment_pool_v2.parquet",
    EVAL_DIR / "heldout_human_judgments_final_v2.parquet",
    EVAL_DIR / "heldout_qrels_final_v2.parquet",
]

for path in candidate_paths:
    print(f"\n{'=' * 20}\n{path.name}")

    if not path.exists():
        print("Not found; skipping.")
        continue

    df = pd.read_parquet(path)
    print("Shape:", df.shape)
    print("Available columns:", df.columns.tolist())

    if "chunk_id" not in df.columns:
        print("No chunk_id column; skipping target lookup.")
        continue

    subset = df.loc[
        df["chunk_id"].astype(str).isin(target_ids)
    ].copy()

    if "query_id" in subset.columns:
        subset = subset.loc[subset["query_id"].eq("H12")]

    if subset.empty:
        print("No matching H12 records.")
        continue

    preferred = [
        "query_id", "chunk_id", "human_relevance",
        "relevance_grade", "bm25_rank", "dense_rank",
        "hybrid_rank", "bm25_score", "dense_score",
        "hybrid_score", "retrieved_by_count", "review_reason",
    ]
    cols = [c for c in preferred if c in subset.columns]

    print(subset[cols].to_string(index=False, max_colwidth=180))

print("\nRead-only inspection complete; no files written.")



heldout_judgment_pool_v2.parquet
Shape: (214, 27)
Available columns: ['query_id', 'corpus_row', 'chunk_id', 'document_id', 'chunk_index', 'text', 'ticker', 'company', 'form_type', 'filing_date', 'report_date', 'accession_number', 'source_url', 'bm25_rank', 'dense_rank', 'hybrid_rank', 'bm25_score', 'dense_score', 'hybrid_score', 'query', 'question_type', 'difficulty', 'report_year', 'retrieved_by_count', 'review_order', 'human_relevance', 'review_reason']
query_id                                chunk_id  human_relevance  bm25_rank  dense_rank  hybrid_rank  bm25_score  dense_score  hybrid_score  retrieved_by_count review_reason
     H12 msft_10k_000095017023035122__chunk_0049             <NA>          6        <NA>         <NA>   16.895615          NaN           NaN                   1              
     H12 msft_10k_000095017023035122__chunk_0055             <NA>       <NA>           7         <NA>         NaN     0.696734           NaN                   1              

heldout_human

In [ ]:
import json
import re
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/03_retrieval_evaluation.ipynb"
)

nb = json.loads(notebook_path.read_text(encoding="utf-8"))

terms = re.compile(
    r"reciprocal.rank|rrf_k|rrf_score|hybrid_score|"
    r"hybrid_rank|bm25_rank|dense_rank|final_top_k|"
    r"candidate_union|rank_fusion",
    re.IGNORECASE,
)

for i, cell in enumerate(nb.get("cells", [])):
    if cell.get("cell_type") != "code":
        continue

    lines = "".join(cell.get("source", [])).splitlines()
    hits = [
        j for j, line in enumerate(lines)
        if terms.search(line)
    ]

    if hits:
        print(f"\n{'=' * 12} CELL {i} {'=' * 12}")
        shown = set()

        for j in hits:
            for k in range(max(0, j - 2), min(len(lines), j + 3)):
                if k not in shown:
                    print(f"{k + 1:>4}: {lines[k][:220]}")
                    shown.add(k)
            print("...")


============ CELL 15 ============
 396: 
 397: # ------------------------------------------------------------
 398: # Hybrid retrieval using Reciprocal Rank Fusion
 399: # ------------------------------------------------------------
 400: 
...
 404:     top_k: int = 10,
 405:     candidate_k: int = 50,
 406:     rrf_k: int = 60,
 407:     ticker=None,
 408:     form_type=None,
...
 410: ) -> pd.DataFrame:
 411:     """
 412:     Fuse BM25 and dense rankings using Reciprocal Rank Fusion.
 413: 
 414:     RRF score:
...
 415:         1 / (rrf_k + BM25 rank)
 416:         + 1 / (rrf_k + dense rank)
 417: 
...
 418:     The same metadata restrictions are applied to both component
...
 447:         )
 448: 
 449:     if not isinstance(rrf_k, int) or rrf_k < 1:
 450:         raise ValueError(
 451:             "rrf_k must be a positive integer."
...
 452:         )
 453: 
...
 476: 
 477:     fused_scores = {}
 478:     bm25_ranks = {}
 479:     dense_ranks = {}
 480: 
...
 481:     for res

In [ ]:
from pathlib import Path
import pandas as pd

ROOT = Path("/content/drive/MyDrive/FinanceRAG-Analyst")
path = ROOT / "artifacts/evaluation/heldout_human_judgments_final_v2.parquet"

df = pd.read_parquet(path)
h12 = df.loc[df["query_id"].eq("H12")].copy()

for col in ["human_relevance", "bm25_rank", "dense_rank", "hybrid_rank"]:
    if col in h12.columns:
        h12[col] = pd.to_numeric(h12[col], errors="coerce")

h12["best_available_rank"] = h12[
    ["bm25_rank", "dense_rank", "hybrid_rank"]
].min(axis=1)

relevant = h12.loc[h12["human_relevance"].ge(1)].copy()

print("H12 total judged chunks:", len(h12))
print("H12 relevant chunks (grade >= 1):", len(relevant))
print("H12 highly relevant chunks (grade = 2):",
      int(h12["human_relevance"].eq(2).sum()))

print("\nRelevant chunks by frozen retrieval method:")
for method, col in [
    ("BM25", "bm25_rank"),
    ("Dense", "dense_rank"),
    ("Hybrid RRF", "hybrid_rank"),
]:
    found = relevant[col].notna()
    in_top10 = relevant[col].le(10)
    print(
        f"{method:12s}: retrieved={int(found.sum())}/{len(relevant)}, "
        f"top10={int(in_top10.sum())}/{len(relevant)}"
    )

cols = [
    "chunk_id", "human_relevance", "bm25_rank",
    "dense_rank", "hybrid_rank", "best_available_rank",
]
cols = [c for c in cols if c in h12.columns]

print("\nAll relevant H12 chunks:")
print(
    relevant.sort_values(
        ["human_relevance", "best_available_rank"],
        ascending=[False, True],
    )[cols].to_string(index=False)
)

print("\nRead-only analysis complete; no files written.")

H12 total judged chunks: 20
H12 relevant chunks (grade >= 1): 4
H12 highly relevant chunks (grade = 2): 2

Relevant chunks by frozen retrieval method:
BM25        : retrieved=1/4, top10=1/4
Dense       : retrieved=3/4, top10=3/4
Hybrid RRF  : retrieved=0/4, top10=0/4

All relevant H12 chunks:
                               chunk_id  human_relevance  bm25_rank  dense_rank  hybrid_rank  best_available_rank
msft_10k_000095017023035122__chunk_0049                2          6        <NA>         <NA>                    6
msft_10k_000095017023035122__chunk_0055                2       <NA>           7         <NA>                    7
msft_10k_000095017023035122__chunk_0047                1       <NA>           8         <NA>                    8
msft_10k_000095017023035122__chunk_0056                1       <NA>          10         <NA>                   10

Read-only analysis complete; no files written.


In [ ]:
# Part 4C — Corrected relevant-evidence loss analysis
# Read-only; no LLM calls or file writes.

from pathlib import Path
import pandas as pd

path = (
    Path("/content/drive/MyDrive/FinanceRAG-Analyst")
    / "artifacts/evaluation/heldout_human_judgments_final_v2.parquet"
)

df = pd.read_parquet(path).copy()

for col in ["human_relevance", "bm25_rank", "dense_rank", "hybrid_rank"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

rel = df.loc[df["human_relevance"].ge(1)].copy()

rel["bm25_top10"] = rel["bm25_rank"].notna() & rel["bm25_rank"].le(10)
rel["dense_top10"] = rel["dense_rank"].notna() & rel["dense_rank"].le(10)
rel["hybrid_top10"] = rel["hybrid_rank"].notna() & rel["hybrid_rank"].le(10)

rel["component_top10"] = rel["bm25_top10"] | rel["dense_top10"]
rel["lost_by_hybrid"] = rel["component_top10"] & ~rel["hybrid_top10"]

summary = rel.groupby("query_id").agg(
    relevant_chunks=("chunk_id", "size"),
    bm25_relevant_top10=("bm25_top10", "sum"),
    dense_relevant_top10=("dense_top10", "sum"),
    hybrid_relevant_top10=("hybrid_top10", "sum"),
    component_relevant_top10=("component_top10", "sum"),
    lost_by_hybrid=("lost_by_hybrid", "sum"),
).reset_index()

print("PER-QUERY SUMMARY")
print(summary.to_string(index=False))

print("\nRELEVANT CHUNKS LOST BY HYBRID")
losses = rel.loc[
    rel["lost_by_hybrid"],
    ["query_id", "chunk_id", "human_relevance",
     "bm25_rank", "dense_rank", "hybrid_rank"],
].sort_values(["query_id", "human_relevance"], ascending=[True, False])

print(losses.to_string(index=False) if not losses.empty else "None")

print("\nAGGREGATE LOST CHUNKS:", int(rel["lost_by_hybrid"].sum()))
print("QUERIES WITH LOSSES:", int(summary["lost_by_hybrid"].gt(0).sum()))

# Explicit consistency check for H12.
h12 = summary.loc[summary["query_id"].eq("H12")]
if not h12.empty:
    print("\nH12 consistency check:")
    print(h12.to_string(index=False))
    assert int(h12.iloc[0]["component_relevant_top10"]) == 4
    assert int(h12.iloc[0]["hybrid_relevant_top10"]) == 0
    assert int(h12.iloc[0]["lost_by_hybrid"]) == 4

print("\nRead-only analysis complete; no files written.")

PER-QUERY SUMMARY
query_id  relevant_chunks  bm25_relevant_top10  dense_relevant_top10  hybrid_relevant_top10  component_relevant_top10  lost_by_hybrid
     H01               11                    7                     6                      6                        11               5
     H02               13                    6                     8                      9                        13               4
     H03                8                    3                     8                      5                         8               3
     H04                9                    6                     4                      5                         9               4
     H05                6                    4                     6                      4                         6               2
     H06                5                    2                     4                      4                         4               1
     H07                1                   

In [ ]:
# Part 4D — Characterize relevant chunks omitted by Hybrid RRF
# Read-only; no files written.

from pathlib import Path
import pandas as pd

path = (
    Path("/content/drive/MyDrive/FinanceRAG-Analyst")
    / "artifacts/evaluation/heldout_human_judgments_final_v2.parquet"
)

df = pd.read_parquet(path).copy()

for col in ["human_relevance", "bm25_rank", "dense_rank", "hybrid_rank"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

rel = df.loc[df["human_relevance"].ge(1)].copy()

rel["bm25_top10"] = rel["bm25_rank"].notna() & rel["bm25_rank"].le(10)
rel["dense_top10"] = rel["dense_rank"].notna() & rel["dense_rank"].le(10)
rel["hybrid_top10"] = rel["hybrid_rank"].notna() & rel["hybrid_rank"].le(10)

lost = rel.loc[
    (rel["bm25_top10"] | rel["dense_top10"])
    & ~rel["hybrid_top10"]
].copy()

lost["source_method"] = lost.apply(
    lambda r: (
        "Both"
        if r["bm25_top10"] and r["dense_top10"]
        else "BM25 only"
        if r["bm25_top10"]
        else "Dense only"
    ),
    axis=1,
)

print("LOST CHUNKS BY HUMAN RELEVANCE GRADE")
print(
    lost.groupby("human_relevance")
    .size()
    .rename("count")
    .to_string()
)

print("\nLOST CHUNKS BY SOURCE RETRIEVER")
print(
    lost.groupby("source_method")
    .size()
    .rename("count")
    .to_string()
)

print("\nCOMPONENT RANK DISTRIBUTION OF LOST CHUNKS")
for col in ["bm25_rank", "dense_rank"]:
    available = lost[col].dropna()
    print(f"\n{col}:")
    if available.empty:
        print("No component ranks available.")
    else:
        print(available.describe().to_string())

print("\nLOST GRADE-2 CHUNKS")
grade2 = lost.loc[
    lost["human_relevance"].eq(2),
    [
        "query_id", "chunk_id", "bm25_rank", "dense_rank",
        "human_relevance", "review_reason",
    ],
]
print(grade2.to_string(index=False))

print("\nRead-only analysis complete; no files written.")

LOST CHUNKS BY HUMAN RELEVANCE GRADE
human_relevance
1    20
2    13

LOST CHUNKS BY SOURCE RETRIEVER
source_method
BM25 only     11
Dense only    22

COMPONENT RANK DISTRIBUTION OF LOST CHUNKS

bm25_rank:
count        11.0
mean     5.272727
std      2.493628
min           2.0
25%           3.0
50%           6.0
75%           6.5
max          10.0

dense_rank:
count        22.0
mean     6.318182
std      2.661657
min           1.0
25%          4.25
50%           6.0
75%          8.75
max          10.0

LOST GRADE-2 CHUNKS
query_id                                chunk_id  bm25_rank  dense_rank  human_relevance review_reason
     H01 aapl_10k_000032019325000079__chunk_0060       <NA>           9                2              
     H01 aapl_10k_000032019325000079__chunk_0108       <NA>           4                2              
     H02 aapl_10k_000032019324000123__chunk_0105       <NA>           4                2              
     H03 msft_10k_000095017025100235__chunk_0081       <NA> 

In [ ]:
# Part 4E — Relevant-evidence gains and losses from hybrid fusion
# Read-only diagnostic; no LLM calls and no files written.

from pathlib import Path
import pandas as pd

path = (
    Path("/content/drive/MyDrive/FinanceRAG-Analyst")
    / "artifacts/evaluation/heldout_human_judgments_final_v2.parquet"
)

df = pd.read_parquet(path).copy()

for col in ["human_relevance", "bm25_rank", "dense_rank", "hybrid_rank"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

rel = df.loc[df["human_relevance"].ge(1)].copy()

for method in ["bm25", "dense", "hybrid"]:
    rel[f"{method}_top10"] = (
        rel[f"{method}_rank"].notna()
        & rel[f"{method}_rank"].le(10)
    )

component_top10 = rel["bm25_top10"] | rel["dense_top10"]

gained = rel.loc[rel["hybrid_top10"] & ~component_top10].copy()
lost = rel.loc[component_top10 & ~rel["hybrid_top10"]].copy()

print("RELEVANT CHUNKS GAINED BY HYBRID")
print(f"Count: {len(gained)}")
print(gained.groupby("human_relevance").size().rename("count").to_string())

print("\nRELEVANT CHUNKS LOST BY HYBRID")
print(f"Count: {len(lost)}")
print(lost.groupby("human_relevance").size().rename("count").to_string())

print("\nNET CHANGE IN RELEVANT CHUNK COVERAGE")
print(f"Hybrid-only gains: {len(gained)}")
print(f"Component-only losses: {len(lost)}")
print(f"Net change: {len(gained) - len(lost)}")

print("\nGAINS BY QUERY")
print(
    gained.groupby("query_id").size()
    .reindex(sorted(df["query_id"].dropna().unique()), fill_value=0)
    .rename("hybrid_only_gains")
    .to_string()
)

print("\nRead-only analysis complete; no files written.")

RELEVANT CHUNKS GAINED BY HYBRID
Count: 2
human_relevance
1    1
2    1

RELEVANT CHUNKS LOST BY HYBRID
Count: 33
human_relevance
1    20
2    13

NET CHANGE IN RELEVANT CHUNK COVERAGE
Hybrid-only gains: 2
Component-only losses: 33
Net change: -31

GAINS BY QUERY
query_id
H01    0
H02    0
H03    0
H04    0
H05    0
H06    1
H07    0
H08    0
H09    0
H10    1
H11    0
H12    0

Read-only analysis complete; no files written.


In [ ]:
# Part 4F — Inspect relevant chunks unique to Hybrid top-10
# Read-only diagnostic; no LLM calls and no files written.

from pathlib import Path
import pandas as pd

path = (
    Path("/content/drive/MyDrive/FinanceRAG-Analyst")
    / "artifacts/evaluation/heldout_human_judgments_final_v2.parquet"
)

df = pd.read_parquet(path).copy()

for col in ["human_relevance", "bm25_rank", "dense_rank", "hybrid_rank"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

rel = df.loc[df["human_relevance"].ge(1)].copy()

rel["bm25_top10"] = rel["bm25_rank"].notna() & rel["bm25_rank"].le(10)
rel["dense_top10"] = rel["dense_rank"].notna() & rel["dense_rank"].le(10)
rel["hybrid_top10"] = rel["hybrid_rank"].notna() & rel["hybrid_rank"].le(10)

gained = rel.loc[
    rel["hybrid_top10"]
    & ~(rel["bm25_top10"] | rel["dense_top10"])
].copy()

columns = [
    c for c in [
        "query_id", "chunk_id", "human_relevance",
        "bm25_rank", "dense_rank", "hybrid_rank", "review_reason",
    ]
    if c in gained.columns
]

print("HYBRID-ONLY RELEVANT CHUNKS")
print(gained[columns].to_string(index=False))
print("\nRead-only analysis complete; no files written.")

HYBRID-ONLY RELEVANT CHUNKS
query_id                                chunk_id  human_relevance  bm25_rank  dense_rank  hybrid_rank review_reason
     H06 amzn_10k_000101872424000008__chunk_0042                1       <NA>        <NA>            7              
     H10 nvda_10k_000104581024000029__chunk_0041                2       <NA>        <NA>           10              

Read-only analysis complete; no files written.


In [ ]:
# Part 4G — Query-level hybrid fusion loss summary
# Read-only; no LLM calls and no files written.

from pathlib import Path
import pandas as pd

path = (
    Path("/content/drive/MyDrive/FinanceRAG-Analyst")
    / "artifacts/evaluation/heldout_human_judgments_final_v2.parquet"
)

df = pd.read_parquet(path).copy()

for col in ["human_relevance", "bm25_rank", "dense_rank", "hybrid_rank"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

rel = df.loc[df["human_relevance"].ge(1)].copy()

for method in ["bm25", "dense", "hybrid"]:
    rel[f"{method}_top10"] = (
        rel[f"{method}_rank"].notna()
        & rel[f"{method}_rank"].le(10)
    )

rel["component_top10"] = rel["bm25_top10"] | rel["dense_top10"]
rel["lost_by_hybrid"] = rel["component_top10"] & ~rel["hybrid_top10"]
rel["gained_by_hybrid"] = rel["hybrid_top10"] & ~rel["component_top10"]

summary = rel.groupby("query_id").agg(
    judged_relevant_chunks=("chunk_id", "size"),
    component_relevant_top10=("component_top10", "sum"),
    hybrid_relevant_top10=("hybrid_top10", "sum"),
    lost_by_hybrid=("lost_by_hybrid", "sum"),
    gained_by_hybrid=("gained_by_hybrid", "sum"),
)

summary["net_difference"] = (
    summary["gained_by_hybrid"] - summary["lost_by_hybrid"]
)

print(summary.sort_values(
    ["lost_by_hybrid", "net_difference"],
    ascending=[False, True],
).to_string())

print("\nRead-only analysis complete; no files written.")

          judged_relevant_chunks  component_relevant_top10  hybrid_relevant_top10  lost_by_hybrid  gained_by_hybrid  net_difference
query_id                                                                                                                           
H01                           11                        11                      6               5                 0              -5
H02                           13                        13                      9               4                 0              -4
H04                            9                         9                      5               4                 0              -4
H12                            4                         4                      0               4                 0              -4
H03                            8                         8                      5               3                 0              -3
H08                            5                         5                  

Part 5 - operational readiness

In [ ]:
# Part 5A — Inspect N5 section headings and cell structure
# Read-only notebook inspection; no LLM calls and no files written.

import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "`05__end_to_end_rag_evaluation_and_deployment_readiness.ipynb`.ipynb"
)

if not notebook_path.exists():
    candidates = list(
        Path("/content/drive/MyDrive/Colab Notebooks").glob("*end_to_end*")
    )
    print("Exact notebook path not found.")
    print("Matching candidates:")
    for candidate in candidates:
        print(candidate)
else:
    nb = json.loads(notebook_path.read_text(encoding="utf-8"))
    print(f"Notebook: {notebook_path.name}")
    print(f"Total cells: {len(nb.get('cells', []))}\n")

    for i, cell in enumerate(nb.get("cells", [])):
        source = "".join(cell.get("source", []))
        first_line = source.strip().splitlines()[0] if source.strip() else ""
        if cell.get("cell_type") == "markdown" or any(
            term in source.lower()
            for term in ["part 5", "operational", "deployment", "readiness"]
        ):
            print(f"[{i:03d}] {cell.get('cell_type', '?')}: {first_line[:180]}")

    print("\nRead-only inspection complete; no files written.")

Notebook: `05__end_to_end_rag_evaluation_and_deployment_readiness.ipynb`.ipynb
Total cells: 29

[000] markdown: Setup
[001] code: # =============================================================================
[008] markdown: Query-level failure analysis

Read-only inspection complete; no files written.


In [ ]:
# Part 5A.1 — Print all N5 cell headings and code comments
# Read-only inspection; no LLM calls and no files written.

import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "`05__end_to_end_rag_evaluation_and_deployment_readiness.ipynb`.ipynb"
)

nb = json.loads(notebook_path.read_text(encoding="utf-8"))

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    lines = [line.strip() for line in source.splitlines() if line.strip()]

    print(f"\n[{i:02d}] {cell.get('cell_type', '?').upper()}")

    if not lines:
        print("(empty)")
        continue

    if cell.get("cell_type") == "markdown":
        print("\n".join(lines[:8]))
    else:
        comments = [
            line for line in lines
            if line.startswith("#") and not line.startswith("# ===")
        ]
        print("\n".join(comments[:8]) if comments else lines[0][:180])

print("\nRead-only inspection complete; no files written.")


[00] MARKDOWN
Setup

[01] CODE
# FINANCERAG-ANALYST — NOTEBOOK 5 PART 0: SETUP AND EVALUATION CONTRACT
# -----------------------------------------------------------------------------
# 0.1 — Mount Google Drive and define project paths
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------
# 0.2 — Validate directory structure
# -----------------------------------------------------------------------------
# -----------------------------------------------------------------------------

[02] CODE
# FINANCERAG-ANALYST — NOTEBOOK 5 PART 1
# ARTIFACT DISCOVERY AND INTEGRITY VALIDATION
#
# Mode: Read-only
# LLM calls: None
# Existing artifact modifications: None
# ------------------------------------------------------------
# 1. Validate Part 0 setup

[03] CODE
# NOTEBOOK 5 — PART 2A
# Cross-artifact query alignment and schema inspection
#
# Prerequisites: Notebook 5 Parts 0 and 1 have run

In [ ]:
# Notebook 5 — Part 5B: Inspect operational-readiness cells
# Read-only inspection; no LLM calls and no files written.

import json
from pathlib import Path

notebook_path = Path(
    "/content/drive/MyDrive/Colab Notebooks/"
    "`05__end_to_end_rag_evaluation_and_deployment_readiness.ipynb`.ipynb"
)

nb = json.loads(notebook_path.read_text(encoding="utf-8"))

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    lines = [line.rstrip() for line in source.splitlines()]

    if any(
        term in source.lower()
        for term in [
            "part 5", "operational readiness", "deployment readiness",
            "part 6", "final report", "handoff"
        ]
    ):
        print(f"\n{'=' * 70}")
        print(f"CELL {i} | {cell.get('cell_type', '?').upper()}")
        print("=" * 70)
        print("\n".join(lines[:100]))

print("\nInspection complete; no files written.")


CELL 5 | CODE

# ============================================================
# NOTEBOOK 5 — PART 3A
# Reproduce aggregate retrieval and generation metrics
#
# Uses frozen Notebook 3 and Notebook 4 outputs.
# No LLM calls. No files written. No source modifications.
# ============================================================

import json
import numpy as np
import pandas as pd
from pathlib import Path

assert "PART2B_UNIFIED" in globals(), "Run Part 2B first."

frames = PART2_FRAMES
metrics = frames["per_query_metrics"].copy()
scores = frames["generation_scores"].copy()
audit = frames["citation_audit"].copy()

# ------------------------------------------------------------
# 1. Retrieval metric reproduction
# ------------------------------------------------------------

retrieval_metric_columns = [
    "useful_mrr_at_10",
    "graded_ndcg_at_10",
    "useful_precision_at_10",
    "useful_pool_recall_at_10",
    "direct_mrr_at_10",
    "graded_ndcg_at_5",
]

available_retrieval_metrics

In [ ]:
# NOTEBOOK 5 — PART 5B
# Operational readiness: inspect existing artifact and completion manifests
# Read-only; no LLM calls; no files written.

import json
from pathlib import Path

print("=" * 76)
print("PART 5B — ARTIFACT AND COMPLETION STATUS")
print("=" * 76)

manifest_paths = {
    "N4 completion manifest": (
        GENERATION_EVALUATION_DIR
        / "notebook_04_completion_manifest_v1.json"
    ),
    "N4 final report": (
        GENERATION_EVALUATION_DIR
        / "notebook_04_final_report_v1.json"
    ),
    "N4 generation evaluation manifest": (
        GENERATION_EVALUATION_DIR
        / "notebook_04_groq_final_generation_evaluation_manifest_v1.json"
    ),
    "N4 citation audit manifest": (
        GENERATION_EVALUATION_DIR
        / "notebook_04_groq_corrected_citation_manifest_v2.json"
    ),
}

for label, path in manifest_paths.items():
    print(f"\n{label}")
    print(f"Path: {path}")
    print(f"Exists: {path.exists()}")

    if not path.exists():
        continue

    try:
        with path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        if isinstance(data, dict):
            for key, value in data.items():
                if isinstance(value, (dict, list)):
                    print(f"  {key}: [{type(value).__name__}, {len(value)} items]")
                else:
                    print(f"  {key}: {value}")
        else:
            print(f"  Root type: {type(data).__name__}")
    except Exception as exc:
        print(f"  Read error: {type(exc).__name__}: {exc}")

print("\n" + "=" * 76)
print("READ-ONLY CHECK COMPLETE — no files written.")

PART 5B — ARTIFACT AND COMPLETION STATUS

N4 completion manifest
Path: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/evaluation/notebook_04_completion_manifest_v1.json
Exists: True
  artifact_name: notebook_04_completion_manifest
  artifact_version: notebook_04_generation_evaluation_v1
  created_at_utc: 2026-08-10T10:54:08.881381+00:00
  status: COMPLETE
  evaluation_scope: [dict, 7 items]
  headline_results: [dict, 22 items]
  failure_analysis: [dict, 5 items]
  known_limitations: [list, 7 items]
  next_engineering_priorities: [list, 4 items]
  reproducibility: [dict, 6 items]

N4 final report
Path: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/evaluation/notebook_04_final_report_v1.json
Exists: True
  notebook: 04_generation_and_grounding_evaluation
  status: COMPLETE
  evaluation_scope: [dict, 7 items]
  headline_results: [dict, 22 items]
  principal_successes: [list, 4 items]
  principal_failures: [list, 3 items]
  conclusion: The evaluated pipelin

In [ ]:
# NOTEBOOK 5 — PART 5C
# Validate artifact references in existing N4 manifests
# Read-only; no LLM calls; no files written.

import json
from pathlib import Path

manifest_paths = {
    "N4 completion manifest": (
        GENERATION_EVALUATION_DIR
        / "notebook_04_completion_manifest_v1.json"
    ),
    "N4 generation evaluation manifest": (
        GENERATION_EVALUATION_DIR
        / "notebook_04_groq_final_generation_evaluation_manifest_v1.json"
    ),
    "N4 citation audit manifest": (
        GENERATION_EVALUATION_DIR
        / "notebook_04_groq_corrected_citation_manifest_v2.json"
    ),
}

def inspect_references(obj, parent_key=""):
    """Collect path-like strings from nested manifest structures."""
    found = []

    if isinstance(obj, dict):
        for key, value in obj.items():
            key_path = f"{parent_key}.{key}" if parent_key else key

            if isinstance(value, str):
                normalized_key = key.lower()
                if (
                    "path" in normalized_key
                    or "file" in normalized_key
                    or "source_artifact" in normalized_key
                    or value.endswith((".json", ".parquet", ".csv", ".jsonl"))
                ):
                    found.append((key_path, value))
            else:
                found.extend(inspect_references(value, key_path))

    elif isinstance(obj, list):
        for i, value in enumerate(obj):
            found.extend(inspect_references(value, f"{parent_key}[{i}]"))

    return found

print("=" * 76)
print("PART 5C — MANIFEST ARTIFACT REFERENCE VALIDATION")
print("=" * 76)

for label, manifest_path in manifest_paths.items():
    print(f"\n--- {label} ---")

    if not manifest_path.exists():
        print("MANIFEST MISSING")
        continue

    with manifest_path.open("r", encoding="utf-8") as f:
        manifest = json.load(f)

    references = inspect_references(manifest)
    if not references:
        print("No path-like artifact references found.")
        continue

    seen = set()
    for key, raw_path in references:
        if raw_path in seen:
            continue
        seen.add(raw_path)

        candidate = Path(raw_path)
        if not candidate.is_absolute():
            candidate = REPO_ROOT / candidate

        print(
            f"{'EXISTS' if candidate.exists() else 'MISSING'}"
            f" | {key} | {candidate}"
        )

print("\nRead-only validation complete; no files written.")

PART 5C — MANIFEST ARTIFACT REFERENCE VALIDATION

--- N4 completion manifest ---
EXISTS | reproducibility.artifact_inventory | /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/evaluation/notebook_04_final_artifact_inventory_v1.csv

--- N4 generation evaluation manifest ---
EXISTS | source_artifact | /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/traces/notebook_04_control_generations_groq_gpt_oss_120b_v2.json
EXISTS | files.manual_scores_parquet | /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/evaluation/notebook_04_groq_manual_answer_scores_v1.parquet
EXISTS | files.manual_scores_json | /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/evaluation/notebook_04_groq_manual_answer_scores_v1.json
EXISTS | files.final_summary | /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/evaluation/notebook_04_groq_final_generation_summary_v1.json

--- N4 citation audit manifest ---
EXISTS | source_generation_traces | /content/dri

In [ ]:
# NOTEBOOK 5 — PART 5D
# Inspect recorded reproducibility status and limitations
# Read-only; no LLM calls; no files written.

import json
from pathlib import Path

manifest_path = (
    GENERATION_EVALUATION_DIR
    / "notebook_04_completion_manifest_v1.json"
)

with manifest_path.open("r", encoding="utf-8") as f:
    manifest = json.load(f)

print("=" * 76)
print("PART 5D — RECORDED REPRODUCIBILITY STATUS")
print("=" * 76)

for section in [
    "reproducibility",
    "known_limitations",
    "next_engineering_priorities",
]:
    print(f"\n--- {section.upper()} ---")
    value = manifest.get(section)

    if isinstance(value, dict):
        for key, item in value.items():
            print(f"{key}: {item}")
    elif isinstance(value, list):
        for i, item in enumerate(value, start=1):
            print(f"{i}. {item}")
    else:
        print(f"Not present; value={value!r}")

print("\nRead-only inspection complete; no files written.")

PART 5D — RECORDED REPRODUCIBILITY STATUS

--- REPRODUCIBILITY ---
raw_generation_traces_modified: False
manual_scores_frozen: True
all_required_artifacts_present: True
artifact_inventory: /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/evaluation/notebook_04_final_artifact_inventory_v1.csv
clean_runtime_execution_verified: False
verification_note: The final artifact inventory passed. A separate clean-runtime execution should be recorded only after the notebook has been restarted and run successfully from top to bottom.

--- KNOWN_LIMITATIONS ---
1. Generation quality was manually evaluated on a small held-out set of 12 questions.
2. The evidence-level rubric was completed by one reviewer; inter-annotator agreement was not measured.
3. The manual rubric scores characterize this frozen evaluation set and must not be presented as universal system accuracy.
4. Four generated answers used Unicode citation variants that failed the initial strict ASCII parser.
5. H01 was accur

In [ ]:
# NOTEBOOK 5 — PART 5E
# Inspect remaining notebook structure and readiness criteria
# Read-only; no LLM calls; no files written.

import json
from pathlib import Path

notebook_path = (
    Path("/content/drive/MyDrive/Colab Notebooks")
    / "`05__end_to_end_rag_evaluation_and_deployment_readiness.ipynb`.ipynb"
)

nb = json.loads(notebook_path.read_text(encoding="utf-8"))

print("=" * 76)
print("NOTEBOOK 5 — REMAINING STRUCTURE")
print("=" * 76)

for i, cell in enumerate(nb.get("cells", [])):
    source = "".join(cell.get("source", []))
    lines = [line.strip() for line in source.splitlines() if line.strip()]

    if not lines:
        continue

    first_line = lines[0]
    if cell.get("cell_type") == "markdown":
        print(f"\n[{i:02d}] MARKDOWN: {first_line[:200]}")
        print("\n".join(lines[1:6]))
    elif any(
        term in source.lower()
        for term in [
            "readiness", "deployment", "restart", "reproducib",
            "part 5", "part 6", "final report", "handoff",
        ]
    ):
        print(f"\n[{i:02d}] CODE")
        print("\n".join(lines[:12]))

print("\nRead-only inspection complete; no files written.")

NOTEBOOK 5 — REMAINING STRUCTURE

[00] MARKDOWN: Setup


[01] CODE
# =============================================================================
# FINANCERAG-ANALYST — NOTEBOOK 5 PART 0: SETUP AND EVALUATION CONTRACT
# =============================================================================
from pathlib import Path
import json
import platform
import sys
import numpy as np
import pandas as pd
from google.colab import drive
# -----------------------------------------------------------------------------
# 0.1 — Mount Google Drive and define project paths

[05] CODE
# ============================================================
# NOTEBOOK 5 — PART 3A
# Reproduce aggregate retrieval and generation metrics
#
# Uses frozen Notebook 3 and Notebook 4 outputs.
# No LLM calls. No files written. No source modifications.
# ============================================================
import json
import numpy as np
import pandas as pd
from pathlib import Path
assert "PART2B_UNIFIED" in globals

In [ ]:
# ============================================================
# NOTEBOOK 5 — PART 5E
# Operational-readiness checklist
#
# Read-only audit of frozen evaluation artifacts and recorded
# reproducibility status. No LLM calls; no files written.
# ============================================================

import json
from pathlib import Path

checks = []

def record_check(name, passed, detail):
    checks.append({
        "check": name,
        "status": "PASS" if passed else "REVIEW",
        "detail": str(detail),
    })

# 1. Confirm required frozen evaluation artifacts exist.
required_artifacts = {
    "N3 retrieval summary": (
        EVALUATION_DIR / "heldout_retrieval_summary_v2.json"
    ),
    "N3 retrieval config": (
        EVALUATION_DIR / "heldout_retrieval_config_v2.json"
    ),
    "N3 per-query metrics": (
        EVALUATION_DIR / "heldout_retrieval_per_query_metrics_v2.parquet"
    ),
    "N3 human judgments": (
        EVALUATION_DIR / "heldout_human_judgments_final_v2.parquet"
    ),
    "N4 completion manifest": (
        GENERATION_EVALUATION_DIR
        / "notebook_04_completion_manifest_v1.json"
    ),
    "N4 final report": (
        GENERATION_EVALUATION_DIR / "notebook_04_final_report_v1.json"
    ),
    "N4 generation summary": (
        GENERATION_EVALUATION_DIR
        / "notebook_04_groq_final_generation_summary_v1.json"
    ),
    "N4 corrected citation audit": (
        GENERATION_EVALUATION_DIR
        / "notebook_04_groq_corrected_citation_audit_v2.parquet"
    ),
}

for name, path in required_artifacts.items():
    record_check(
        f"Artifact exists: {name}",
        path.exists(),
        path if path.exists() else f"MISSING: {path}",
    )

# 2. Inspect recorded Notebook 4 completion status.
completion_path = (
    GENERATION_EVALUATION_DIR
    / "notebook_04_completion_manifest_v1.json"
)

if completion_path.exists():
    with completion_path.open("r", encoding="utf-8") as f:
        completion = json.load(f)

    record_check(
        "N4 completion manifest status",
        completion.get("status") == "COMPLETE",
        completion.get("status", "Status field missing"),
    )

    reproducibility = completion.get("reproducibility", {})

    record_check(
        "N4 frozen manual scores",
        reproducibility.get("manual_scores_frozen") is True,
        reproducibility.get("manual_scores_frozen"),
    )

    record_check(
        "N4 raw generation traces unchanged",
        reproducibility.get("raw_generation_traces_modified") is False,
        reproducibility.get("raw_generation_traces_modified"),
    )

    clean_run_verified = (
        reproducibility.get("clean_runtime_execution_verified") is True
    )
    record_check(
        "N4 clean-runtime execution verified",
        clean_run_verified,
        reproducibility.get(
            "verification_note",
            f"Recorded value: {clean_run_verified}",
        ),
    )
else:
    record_check(
        "N4 completion manifest readable",
        False,
        "Completion manifest is missing; cannot inspect status.",
    )

# 3. Report whether the current N5 runtime has the expected core objects.
for object_name in [
    "PART2B_UNIFIED",
    "PART3A_METRICS",
]:
    exists_in_runtime = object_name in globals()
    record_check(
        f"N5 runtime object: {object_name}",
        exists_in_runtime,
        "Present in current runtime" if exists_in_runtime
        else "Not found; this may reflect naming differences or execution order",
    )

# 4. Display results.
print("=" * 78)
print("NOTEBOOK 5 — PART 5E: OPERATIONAL READINESS")
print("=" * 78)

for item in checks:
    print(f"\n[{item['status']}] {item['check']}")
    print(f"  {item['detail']}")

passed = sum(item["status"] == "PASS" for item in checks)
review = sum(item["status"] == "REVIEW" for item in checks)

print("\n" + "=" * 78)
print(f"CHECKS PASSED: {passed}/{len(checks)}")
print(f"ITEMS REQUIRING REVIEW: {review}")
print("Audit only — no files written and no LLM calls made.")

NOTEBOOK 5 — PART 5E: OPERATIONAL READINESS

[PASS] Artifact exists: N3 retrieval summary
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_summary_v2.json

[PASS] Artifact exists: N3 retrieval config
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_config_v2.json

[PASS] Artifact exists: N3 per-query metrics
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_per_query_metrics_v2.parquet

[PASS] Artifact exists: N3 human judgments
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_human_judgments_final_v2.parquet

[PASS] Artifact exists: N4 completion manifest
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/evaluation/notebook_04_completion_manifest_v1.json

[PASS] Artifact exists: N4 final report
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/generation/evaluation/notebook_04_final_report_v1.json

[PASS] Artifact exists: N4 generation summary

In [ ]:
# ============================================================
# NOTEBOOK 5 — PART 5G
# Definitive Notebook 4 artifact and consistency audit
# Read-only. No LLM calls. No files written.
# ============================================================

import json
import hashlib
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path("/content/drive/MyDrive/FinanceRAG-Analyst")
ARTIFACTS = PROJECT_ROOT / "artifacts"
N3_DIR = ARTIFACTS / "evaluation"
N4_DIR = ARTIFACTS / "generation" / "evaluation"
TRACE_PATH = (
    ARTIFACTS / "generation" / "traces"
    / "notebook_04_control_generations_groq_gpt_oss_120b_v2.json"
)

checks = []

def audit_check(name, passed, detail):
    checks.append({
        "check": name,
        "status": "PASS" if passed else "REVIEW",
        "detail": str(detail),
    })

def load_json(path):
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

# 1. Required frozen artifacts
required_files = {
    "N3 retrieval summary":
        N3_DIR / "heldout_retrieval_summary_v2.json",
    "N3 retrieval config":
        N3_DIR / "heldout_retrieval_config_v2.json",
    "N3 per-query metrics":
        N3_DIR / "heldout_retrieval_per_query_metrics_v2.parquet",
    "N3 human judgments":
        N3_DIR / "heldout_human_judgments_final_v2.parquet",
    "N4 completion manifest":
        N4_DIR / "notebook_04_completion_manifest_v1.json",
    "N4 final report":
        N4_DIR / "notebook_04_final_report_v1.json",
    "N4 generation manifest":
        N4_DIR / "notebook_04_groq_final_generation_evaluation_manifest_v1.json",
    "N4 generation summary":
        N4_DIR / "notebook_04_groq_final_generation_summary_v1.json",
    "N4 manual scores parquet":
        N4_DIR / "notebook_04_groq_manual_answer_scores_v1.parquet",
    "N4 corrected citation audit":
        N4_DIR / "notebook_04_groq_corrected_citation_audit_v2.parquet",
    "N4 corrected citation manifest":
        N4_DIR / "notebook_04_groq_corrected_citation_manifest_v2.json",
    "N4 raw generation traces":
        TRACE_PATH,
}

for label, path in required_files.items():
    audit_check(
        f"Exists: {label}",
        path.is_file(),
        path if path.is_file() else f"Missing: {path}",
    )

# 2. Load manifests and summaries if present
loaded = {}
for label, path in required_files.items():
    if not path.is_file():
        continue
    try:
        if path.suffix == ".json":
            loaded[label] = load_json(path)
        elif path.suffix == ".parquet":
            loaded[label] = pd.read_parquet(path)
    except Exception as e:
        audit_check(f"Readable: {label}", False, repr(e))
    else:
        audit_check(f"Readable: {label}", True, "Loaded successfully")

# 3. Completion manifest claims
completion = loaded.get("N4 completion manifest", {})
if isinstance(completion, dict):
    audit_check(
        "N4 recorded status is COMPLETE",
        completion.get("status") == "COMPLETE",
        completion.get("status"),
    )

    repro = completion.get("reproducibility", {})
    audit_check(
        "Raw generation traces recorded unchanged",
        repro.get("raw_generation_traces_modified") is False,
        repro.get("raw_generation_traces_modified"),
    )
    audit_check(
        "Manual scores recorded frozen",
        repro.get("manual_scores_frozen") is True,
        repro.get("manual_scores_frozen"),
    )
    audit_check(
        "Fresh clean-runtime run verified",
        repro.get("clean_runtime_execution_verified") is True,
        repro.get(
            "verification_note",
            repro.get("clean_runtime_execution_verified"),
        ),
    )

# 4. Expected sample sizes and cross-artifact row counts
expected_rows = {
    "N3 per-query metrics": 12,
    "N4 manual scores parquet": 12,
    "N4 corrected citation audit": 12,
}

for label, expected in expected_rows.items():
    obj = loaded.get(label)
    if isinstance(obj, pd.DataFrame):
        audit_check(
            f"Expected row count: {label}",
            len(obj) == expected,
            f"rows={len(obj)}, expected={expected}",
        )

# 5. Compare query identifiers across N4 tabular artifacts, where available
def find_query_column(df):
    candidates = [
        "query_id", "question_id", "heldout_query_id", "case_id", "qid"
    ]
    for col in candidates:
        if col in df.columns:
            return col
    return None

n4_tables = [
    "N4 manual scores parquet",
    "N4 corrected citation audit",
]
query_sets = {}

for label in n4_tables:
    df = loaded.get(label)
    if isinstance(df, pd.DataFrame):
        col = find_query_column(df)
        if col is not None:
            query_sets[label] = set(df[col].dropna().astype(str))
            print(f"{label}: query column={col}, "
                  f"unique queries={len(query_sets[label])}")
        else:
            print(f"{label}: no recognized query-ID column; "
                  "query-level comparison skipped")

if len(query_sets) == 2:
    labels = list(query_sets)
    same_ids = query_sets[labels[0]] == query_sets[labels[1]]
    audit_check(
        "N4 manual scores and citation audit query IDs align",
        same_ids,
        f"{labels[0]}={len(query_sets[labels[0]])} IDs; "
        f"{labels[1]}={len(query_sets[labels[1]])} IDs",
    )

# 6. SHA-256 fingerprints for key frozen files (no file changes)
fingerprint_paths = [
    N4_DIR / "notebook_04_completion_manifest_v1.json",
    N4_DIR / "notebook_04_groq_manual_answer_scores_v1.parquet",
    N4_DIR / "notebook_04_groq_corrected_citation_audit_v2.parquet",
    TRACE_PATH,
]

print("\n" + "=" * 78)
print("FROZEN ARTIFACT FINGERPRINTS")
print("=" * 78)
for path in fingerprint_paths:
    if path.is_file():
        digest = hashlib.sha256(path.read_bytes()).hexdigest()
        print(f"{path.name}\n  SHA256: {digest}")

# 7. Final audit summary
print("\n" + "=" * 78)
print("NOTEBOOK 4 — DEFINITIVE ARTIFACT AUDIT")
print("=" * 78)

for item in checks:
    print(f"\n[{item['status']}] {item['check']}")
    print(f"  {item['detail']}")

passed = sum(x["status"] == "PASS" for x in checks)
review = sum(x["status"] == "REVIEW" for x in checks)

print("\n" + "-" * 78)
print(f"PASS: {passed} | REVIEW: {review} | TOTAL: {len(checks)}")
print("Read-only audit complete. No LLM calls. No files written.")

N4 manual scores parquet: query column=query_id, unique queries=12
N4 corrected citation audit: query column=query_id, unique queries=12

FROZEN ARTIFACT FINGERPRINTS
notebook_04_completion_manifest_v1.json
  SHA256: 5fb6f5709d2520fe5281f79f6d5a561744fb7b7916c67aef285f263ccb8295e7
notebook_04_groq_manual_answer_scores_v1.parquet
  SHA256: c7a2affe5d75f9dba3408b39a886c330717a1f6d0c5ca8ac13ee9e8b5bf180c8
notebook_04_groq_corrected_citation_audit_v2.parquet
  SHA256: 3c4cf6685e7ea82caf8fa6730d9698315bd1d69dcb042bf1ec55f90dfd5b7ad4
notebook_04_control_generations_groq_gpt_oss_120b_v2.json
  SHA256: 42b4bc3a8317bf36738b773614c0d5ef3488ec2ca8b80ea7a17d8bc70c5f87ee

NOTEBOOK 4 — DEFINITIVE ARTIFACT AUDIT

[PASS] Exists: N3 retrieval summary
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_summary_v2.json

[PASS] Exists: N3 retrieval config
  /content/drive/MyDrive/FinanceRAG-Analyst/artifacts/evaluation/heldout_retrieval_config_v2.json

[PASS] Exists: N3 per-

In [ ]:
# ============================================================
# NOTEBOOK 5 — PART 6
# Final end-to-end evaluation report and handoff
#
# No LLM calls. No files written. Frozen artifacts unchanged.
# ============================================================

from datetime import datetime, timezone
import json

FINAL_N5_REPORT = {
    "report_name": "FinanceRAG-Analyst End-to-End Evaluation",
    "report_version": "v1",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "evaluation_scope": {
        "retrieval_questions": 12,
        "human_relevance_judgments": 214,
        "generation_questions": 12,
        "retrieval_artifacts_frozen": True,
        "generation_artifacts_frozen": True,
        "new_llm_calls_made": False,
    },
    "retrieval_results": {
        "direct_mrr_at_10": {
            "BM25": 0.494444,
            "Dense": 0.451025,
            "Hybrid_RRF": 0.673611,
        },
        "graded_ndcg_at_10": {
            "BM25": 0.511835,
            "Dense": 0.582095,
            "Hybrid_RRF": 0.598620,
        },
        "pool_relative_useful_recall_at_10": {
            "BM25": 0.568575,
            "Dense": 0.745579,
            "Hybrid_RRF": 0.607082,
        },
        "interpretation": (
            "Hybrid RRF has the strongest direct MRR@10 and graded nDCG@10 "
            "on this held-out set. Dense retrieval has the strongest "
            "pool-relative useful recall. These metrics capture different "
            "retrieval properties and should be reported together."
        ),
    },
    "generation_results": {
        "structured_responses_valid": "12/12",
        "answer_correctness": 0.875,
        "evidence_support": 0.9583,
        "completeness": 0.875,
        "citation_correctness": 0.9583,
        "overall_rubric_score": "88/96 (91.67%)",
        "acceptable_answers": "11/12 (91.67%)",
        "fully_passing_answers": "9/12 (75%)",
        "material_failures": 1,
        "citation_note": (
            "Unicode-aware normalization recognized citations in 12/12 "
            "answers; this measures citation-format recognition, not "
            "whether every cited passage semantically supports the claim."
        ),
    },
    "failure_analysis": {
        "primary_material_failure": "H12",
        "failure_type": "Retrieval-led answer failure",
        "description": (
            "The Microsoft 2023 Form 10-K cybersecurity-risk question "
            "did not receive the central cybersecurity disclosure in its "
            "selected context. Relevant chunks appeared in component "
            "retrieval results but fell outside the hybrid top-10 selection."
        ),
        "broader_retrieval_diagnostic": (
            "Across 12 queries, 33 judged-relevant query/chunk instances "
            "appeared in BM25 or dense top-10 results but not hybrid top-10; "
            "two relevant instances were hybrid-only. This is a pooled-"
            "judgment diagnostic, not an exhaustive corpus-recall estimate."
        ),
    },
    "operational_readiness": {
        "required_artifact_presence_and_readability": "PASS",
        "n4_completion_manifest": "COMPLETE",
        "n4_manual_scores_frozen": True,
        "n4_raw_traces_recorded_unchanged": True,
        "manual_score_and_citation_audit_query_ids_align": True,
        "fresh_clean_runtime_execution": "NOT VERIFIED",
        "n3_per_query_row_count": (
            "36 rows are consistent with 12 questions evaluated across "
            "3 retrieval methods; the earlier checklist expectation of "
            "12 rows was incorrect."
        ),
    },
    "limitations": [
        "The held-out evaluation contains only 12 questions.",
        "Manual generation scoring used one reviewer; inter-annotator "
        "agreement was not measured.",
        "Retrieval relevance is based on a pooled set of judged chunks, "
        "not exhaustive judgments over the entire corpus.",
        "H12 is a material retrieval-led failure and must remain visible "
        "in any project demonstration or evaluation summary.",
        "Fresh restart-and-run-all reproducibility of Notebook 4 has not "
        "been verified in this audit.",
        "Citation-label validity does not establish semantic entailment.",
    ],
    "recommended_next_steps": [
        "Complete a clean-runtime verification only when explicitly "
        "needed, using saved checkpoints and guarding against unexpected "
        "LLM regeneration.",
        "Package the current pipeline and build a Streamlit demo using "
        "the frozen baseline.",
        "Keep H12 as a regression test for cybersecurity-risk retrieval.",
        "Run retrieval experiments separately from the frozen held-out "
        "evaluation before claiming improvements.",
        "Expand the held-out question set and introduce a second reviewer "
        "before making broader quality claims.",
    ],
    "readiness_decision": (
        "READY FOR CONTROLLED PACKAGING AND DEMO DEVELOPMENT, with "
        "documented evaluation limitations. Not yet certified as clean-run "
        "reproducible, and not evidence of production-grade reliability."
    ),
}

print("=" * 78)
print("NOTEBOOK 5 — FINAL EVALUATION REPORT")
print("=" * 78)
print(json.dumps(FINAL_N5_REPORT, indent=2, ensure_ascii=False))

NOTEBOOK 5 — FINAL EVALUATION REPORT
{
  "report_name": "FinanceRAG-Analyst End-to-End Evaluation",
  "report_version": "v1",
  "generated_at_utc": "2026-10-22T08:55:28.441125+00:00",
  "evaluation_scope": {
    "retrieval_questions": 12,
    "human_relevance_judgments": 214,
    "generation_questions": 12,
    "retrieval_artifacts_frozen": true,
    "generation_artifacts_frozen": true,
    "new_llm_calls_made": false
  },
  "retrieval_results": {
    "direct_mrr_at_10": {
      "BM25": 0.494444,
      "Dense": 0.451025,
      "Hybrid_RRF": 0.673611
    },
    "graded_ndcg_at_10": {
      "BM25": 0.511835,
      "Dense": 0.582095,
      "Hybrid_RRF": 0.59862
    },
    "pool_relative_useful_recall_at_10": {
      "BM25": 0.568575,
      "Dense": 0.745579,
      "Hybrid_RRF": 0.607082
    },
    "interpretation": "Hybrid RRF has the strongest direct MRR@10 and graded nDCG@10 on this held-out set. Dense retrieval has the strongest pool-relative useful recall. These metrics capture differ

inspection

In [ ]:
# ============================================================
# FINANCERAG-ANALYST — STREAMLIT PREPARATION
# Step 1: Inspect project structure
# Read-only; no LLM calls; no files written.
# ============================================================

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/FinanceRAG-Analyst")

assert ROOT.exists(), f"Project root not found: {ROOT}"

print("=" * 76)
print("PROJECT ROOT")
print(ROOT)
print("=" * 76)

skip_dirs = {
    ".git", "__pycache__", ".venv", "venv",
    "node_modules", ".ipynb_checkpoints"
}

for path in sorted(ROOT.iterdir()):
    if path.is_dir():
        print(f"\n[DIR]  {path.name}/")
    else:
        print(f"[FILE] {path.name}")

print("\n" + "=" * 76)
print("RELEVANT SOURCE FILES")
print("=" * 76)

source_extensions = {".py", ".toml", ".yaml", ".yml", ".md", ".txt"}
matches = []

for path in ROOT.rglob("*"):
    if not path.is_file():
        continue

    relative = path.relative_to(ROOT)

    if any(part in skip_dirs for part in relative.parts):
        continue

    if any(part.lower() in {
        "artifacts", "data", "corpus", "indexes", "index"
    } for part in relative.parts[:-1]):
        continue

    if path.suffix.lower() in source_extensions:
        matches.append(relative)

for path in sorted(matches):
    print(path)

print("\n" + "=" * 76)
print("EXISTING APP / DEPENDENCY FILES")
print("=" * 76)

important_names = {
    "app.py", "streamlit_app.py", "requirements.txt",
    "pyproject.toml", "setup.py", "environment.yml",
    "readme.md", ".gitignore"
}

for path in sorted(ROOT.rglob("*")):
    if not path.is_file():
        continue

    relative = path.relative_to(ROOT)

    if any(part in skip_dirs for part in relative.parts):
        continue

    if path.name.lower() in important_names:
        print(relative)



PROJECT ROOT
/content/drive/MyDrive/FinanceRAG-Analyst

[DIR]  artifacts/

RELEVANT SOURCE FILES

EXISTING APP / DEPENDENCY FILES


In [ ]:
# ============================================================
# FINANCERAG-ANALYST — STREAMLIT PREPARATION
# Step 2: Inspect artifact structure
# Read-only; no LLM calls; no files written.
# ============================================================

from pathlib import Path

ROOT = Path("/content/drive/MyDrive/FinanceRAG-Analyst")
ARTIFACTS = ROOT / "artifacts"

print("=" * 76)
print("ARTIFACT DIRECTORY STRUCTURE")
print("=" * 76)

for path in sorted(ARTIFACTS.rglob("*")):
    relative = path.relative_to(ROOT)

    if path.is_dir():
        print(f"[DIR]  {relative}/")
    else:
        size_kb = path.stat().st_size / 1024
        print(f"[FILE] {relative}  ({size_kb:,.1f} KB)")

print("\n" + "=" * 76)
print("FILE COUNTS BY EXTENSION")
print("=" * 76)

from collections import Counter

files = [p for p in ARTIFACTS.rglob("*") if p.is_file()]
counts = Counter(p.suffix.lower() or "[no extension]" for p in files)

for extension, count in sorted(counts.items()):
    print(f"{extension}: {count}")

print(f"\nTotal files: {len(files)}")

ARTIFACT DIRECTORY STRUCTURE
[DIR]  artifacts/evaluation/
[FILE] artifacts/evaluation/annotation_progress.csv  (0.2 KB)
[FILE] artifacts/evaluation/annotation_rubric.json  (1.4 KB)
[FILE] artifacts/evaluation/gemini_judge_calibration_20_checkpoint.parquet  (9.2 KB)
[FILE] artifacts/evaluation/groq_judge_calibration_20.csv  (6.9 KB)
[FILE] artifacts/evaluation/groq_judge_calibration_20.parquet  (12.7 KB)
[FILE] artifacts/evaluation/groq_judge_calibration_20_checkpoint.parquet  (12.2 KB)
[FILE] artifacts/evaluation/heldout_human_judgments_final_v2.csv  (553.3 KB)
[FILE] artifacts/evaluation/heldout_human_judgments_final_v2.parquet  (231.0 KB)
[FILE] artifacts/evaluation/heldout_human_judgments_final_v2_manifest.json  (0.9 KB)
[FILE] artifacts/evaluation/heldout_judgment_pool_v2.csv  (552.9 KB)
[FILE] artifacts/evaluation/heldout_judgment_pool_v2.parquet  (230.6 KB)
[FILE] artifacts/evaluation/heldout_judgments_working_v2.csv  (553.3 KB)
[FILE] artifacts/evaluation/heldout_judgments_worki